# Part 2 — Relating racialised discourse to socioeconomic/political data

**Scope**: descriptive, correlational, and predictive — Spearman correlations with multiple-comparison correction, non-parametric group comparisons, proper spatial autocorrelation, and (Part 3, §3.1 onward) a negative-binomial regression model, run once the IAA/robustness work established how much to trust the `racialised` label itself.

## Leads from Part 1 (`descriptive_analysis.ipynb`) carried into this pass

Part 1's closing "Open threads" section flagged several leads that need socioeconomic/political data to resolve. This pass picks up one directly:

- **Arab ethnicity-share** (§2.4g below): the only `ethnicity_group4` share that survives controlling for `population_density_per_km2` (rho=+0.132 vs. raw +0.211) — `Germany`'s link disappears entirely under that control and `Türkiye`'s weakens to non-significance. That check only ever controlled for density; it has not yet been run against this notebook's own deprivation/political predictors.

**Deliberately not pursued in this pass** (so scope stays bounded):
- The `Staff`-theme rural-concentration puzzle — Part 1 resolved it as a genuine within-group effect, not a compositional one, but *why* places differ remains an open, qualitative lead, not something a socioeconomic correlate is likely to explain.
- `approach` has near-chance inter-annotator agreement (κ≈0-0.09) in Part 1 — carried forward here only as a caution, not re-tested, wherever `approach` would otherwise appear (it doesn't feature in this notebook's own crosstabs, so this is a non-issue in practice, noted for completeness).


# Part 0 — Data validation

Before treating any socioeconomic/political predictor as informative for §2.4 onward's correlations, this Part checks two things end-to-end: (1) that the raw source data survived cleaning without corruption, and (2) that the derivations this notebook performs on top of that cleaned data (raw total/rate → per-capita or percentage predictor) actually preserve real information rather than just repackaging it. This was prompted directly by a supervisor note: confirm the data isn't corrupt, and confirm that what we do with the substantive predictor columns is actually making a difference. In scope: raw columns 7–48 of `federal_baseline_eng.csv` -- everything from `Population – Germans (in 1,000s)` through `Unemployment rate November 2024 – men` -- i.e. the whole predictor-ingredient column set, excluding only the identifying columns (1–3), municipalities/area (4–5), total population itself (6), and the unemployment-breakdown tail (49–51).

**What's below, in order:** §0.1 checks the raw-to-cleaned pipeline directly (decimal-comma parsing, `(in 1,000s)` scaling, range sanity, unexpected nulls), using `federal_baseline_eng.csv`/`federal_baseline_eng_clean.csv` before any of this notebook's own derivations touch the data. §2.1–§2.3b then build this notebook's own analysis table and predictor set -- kept under their original numbering since later sections (§2.4 onward) cross-reference them by that number -- and include an existing denominator/merge-integrity audit (§2.3b), promoted here to sit before any correlation work rather than mid-notebook where it was added earlier. §0.2 closes Part 0 by checking whether this notebook's rate→count→rate/per-capita derivations actually add information over the originally-published rates, folding in §2.3b's Berlin/Hamburg finding as the sharpest case where they don't. Everything from `# Correlations, relationships, and spatial analysis` onward assumes Part 0 has already passed.

## 0. Setup — continuing from Part 1

This notebook continues directly from `descriptive_analysis.ipynb`, which built and verified the place→constituency spatial join and the constituency-level aggregate table. Rather than repeating that join, we load its two saved outputs (`const_agg.parquet`, `racialised_reviews_geo.parquet`) plus the socioeconomic/political CSV and the constituency shapefile (needed again here only for its polygon geometry, to draw maps and compute spatial autocorrelation — not to redo the point-in-polygon join itself).

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)

ANALYSIS_DIR = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/analysis_10072026")
SOCIO_DIR = ANALYSIS_DIR / "socio-economic_data"
OUTPUTS_DIR = ANALYSIS_DIR / "notebook_outputs"
FIGURES_DIR = ANALYSIS_DIR / "figures"
PROJECT_ROOT = Path("/home/ethel/Desktop/swimming_pool_project")
SHAPEFILE_PATH = PROJECT_ROOT / "btw25_geometrie_wahlkreise_shp" / "btw25_geometrie_wahlkreise_shp.shp"

political = pd.read_csv(SOCIO_DIR / "federal_baseline_eng_selected_political.csv")
const_agg = pd.read_parquet(OUTPUTS_DIR / "const_agg.parquet")
racialised_df = pd.read_parquet(OUTPUTS_DIR / "racialised_reviews_geo.parquet")

print(f"political/socioeconomic: {political.shape[0]} constituencies x {political.shape[1]} cols")
print(f"const_agg (from Part 1): {const_agg.shape[0]} constituencies x {const_agg.shape[1]} cols")
print(f"racialised_df (from Part 1): {racialised_df.shape[0]} reviews x {racialised_df.shape[1]} cols")

assert const_agg.shape[0] == 299, "expected 299 constituencies"
assert racialised_df.shape[0] == 7_768, "racialised review count drifted from Part 1"


## 0.1 Corruption checks across the raw socioeconomic columns (cols 7–48)

`clean_federal_baseline_eng.py` converts German decimal commas to dots, multiplies columns marked `(in 1,000s)` by 1000, and renames headers to snake_case -- but that script's own logic has never been checked against the actual raw strings before now. This section tests it directly rather than trusting it: decimal-comma parsing (with an explicit check for thousands-separator confusion, since Germany can also use `.` as a thousands separator), the `×1000` scaling multiplier, post-cleaning range sanity for every `_pct`/count/per-1,000 column, and unexpected nulls -- across the 42 substantive columns spanning raw columns 7–48.

In [ ]:
# 0.1a -- decimal-comma parsing + "(in 1,000s)" scaling multiplier checks, run directly against
# the raw strings in federal_baseline_eng.csv (not the already-cleaned file), so this isn't just
# trusting clean_federal_baseline_eng.py's own logic back at itself
from decimal import Decimal

raw_socio = pd.read_csv(SOCIO_DIR / "federal_baseline_eng.csv", encoding="utf-8-sig", dtype=str)
clean_socio_full = pd.read_csv(SOCIO_DIR / "federal_baseline_eng_clean.csv")
TOTAL_ROW_NAMES = {"Land insgesamt", "Insgesamt"}
raw_const = raw_socio.loc[~raw_socio["constituency"].isin(TOTAL_ROW_NAMES)].reset_index(drop=True)
clean_const = clean_socio_full.loc[~clean_socio_full["constituency"].isin(TOTAL_ROW_NAMES)].reset_index(drop=True)
assert (raw_const["constituency_code"].values == clean_const["constituency_code"].astype(str).values).all()
assert raw_const.shape[0] == clean_const.shape[0] == 299

RAW_COLS_7_48 = raw_socio.columns[6:48].tolist()  # 1-indexed cols 7-48: the predictor-ingredient set
print(f"{len(RAW_COLS_7_48)} raw columns in scope: {RAW_COLS_7_48[0]!r} ... {RAW_COLS_7_48[-1]!r}")


def parse_number(value):
    """Mirrors clean_federal_baseline_eng.py's own parse_number, tested independently here."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    normalized = str(value).strip().replace(",", ".")
    if not normalized:
        return None
    try:
        return Decimal(normalized)
    except Exception:
        return None


# spot-check: raw string -> parsed value, for a handful of rows across a few representative columns
spot_check_cols = [
    "Population as of 31.12.2023 – Germans (in 1,000s)",
    "Population as of 31.12.2023 – Foreign nationals (%)",
    "Disposable income of private households 2021 (EUR per inhabitant)",
    "Unemployment rate November 2024 – total",
]
spot_rows = [0, 50, 120, 200, 260]
print("\nspot-check raw string -> parsed float (comma-as-decimal):")
for col in spot_check_cols:
    for i in spot_rows[:2]:
        raw_val = raw_const.loc[i, col]
        parsed = parse_number(raw_val)
        print(f"  {col[:55]:55s} row {i:>3}: raw={raw_val!r:>10} -> parsed={float(parsed) if parsed is not None else None}")

# thousands-separator confusion check: any raw cell across cols 7-48 containing a literal '.' would
# mean the naive comma->dot replace in parse_number could be corrupting a German thousands-dot,
# e.g. mangling '12.345' (twelve thousand three hundred forty-five) into 12.345
n_dot_cells = sum(
    raw_const[col].dropna().astype(str).str.contains(r"\.", regex=True).sum() for col in RAW_COLS_7_48
)
print(f"\ncells in cols 7-48 containing a literal '.' (thousands-separator risk): {n_dot_cells} "
      f"({'PASS -- no ambiguity possible' if n_dot_cells == 0 else 'NEEDS REVIEW'})")

# scaling multiplier: the two "(in 1,000s)" columns (total + German population) should be raw x 1000
THOUSAND_COLS = {
    "Population as of 31.12.2023 – Total (in 1,000s)": "total_population",
    "Population as of 31.12.2023 – Germans (in 1,000s)": "german_population",
}
print("\nscaling multiplier check (raw value x 1000 == cleaned value):")
for raw_col, clean_col in THOUSAND_COLS.items():
    raw_vals = raw_const[raw_col].str.replace(",", ".").astype(float) * 1000
    clean_vals = clean_const[clean_col].astype(float)
    max_diff = (raw_vals - clean_vals).abs().max()
    print(f"  {clean_col:20s} max abs diff after x1000: {max_diff:.2e}  {'PASS' if max_diff < 1e-6 else 'FAIL'}")

# and confirm the multiplier was NOT accidentally applied to columns that aren't marked (in 1,000s)
NOT_SCALED_CHECK = {
    "Population as of 31.12.2023 – Foreign nationals (%)": "foreign_population_pct",
    "Unemployment rate November 2024 – total": "unemployment_total",
    "Business register 2022 – total enterprises (per 1,000 inhabitants)": "enterprises_per_1k",
}
for raw_col, clean_col in NOT_SCALED_CHECK.items():
    raw_vals = raw_const[raw_col].str.replace(",", ".").astype(float)
    clean_vals = clean_const[clean_col].astype(float)
    max_diff = (raw_vals - clean_vals).abs().max()
    print(f"  {clean_col:20s} max abs diff, unscaled expected: {max_diff:.2e}  {'PASS' if max_diff < 1e-6 else 'FAIL'}")


In [ ]:
# 0.1b -- post-cleaning range sanity and unexpected-null checks, across every column derived from
# raw cols 7-48, using the full cleaned file (federal_baseline_eng_clean.csv, 299 constituencies
# after excluding the 16 state-total + 1 Germany-total rows)
checks_01 = []

pct_cols = [c for c in clean_const.columns if c.endswith("_pct")]
for c in pct_cols:
    vals = clean_const[c].astype(float)
    ok = vals.min() >= 0 and vals.max() <= 100
    checks_01.append(("range: [0, 100]", c, f"{vals.min():.2f} to {vals.max():.2f}", ok))

count_cols = [
    c for c in clean_const.columns
    if c.endswith(("_population", "_total"))
    or c in ("total_population", "german_population", "foreign_population")
]
count_cols = list(dict.fromkeys(count_cols))
for c in count_cols:
    vals = pd.to_numeric(clean_const[c], errors="coerce")
    ok = vals.min() >= 0
    checks_01.append(("range: non-negative", c, f"min={vals.min():.0f}", ok))

per1k_plausible_ranges = {
    "employees_per_1k": (0, 1000), "sgb_ii_recipients_per_1k": (0, 500), "cars_per_1k": (0, 1000),
    "enterprises_per_1k": (0, 200), "craft_enterprises_per_1k": (0, 50),
    "completed_dwellings_per_1k": (0, 30), "housing_stock_per_1k": (0, 1000),
    "school_graduates_per_1k": (0, 30), "unemployment_total": (0, 30), "unemployment_men": (0, 30),
    "unemployment_women": (0, 30), "unemployment_age_15_24": (0, 30), "unemployment_age_55_64": (0, 30),
}
for c, (lo_bound, hi_bound) in per1k_plausible_ranges.items():
    vals = pd.to_numeric(clean_const[c], errors="coerce")
    ok = vals.min() >= lo_bound and vals.max() <= hi_bound
    checks_01.append((f"plausible range [{lo_bound}, {hi_bound}]", c, f"{vals.min():.2f} to {vals.max():.2f}", ok))

# unexpected nulls: every one of the cleaned columns should be fully populated for all 299
# constituencies (no silently-dropped parse failure)
na_counts = clean_const.isna().sum()
n_cols_with_na = int((na_counts > 0).sum())
checks_01.append((
    "no unexpected nulls (299 rows)", f"all {clean_const.shape[1]} cleaned columns",
    f"{n_cols_with_na} columns with any NaN", n_cols_with_na == 0,
))

# raw parse failures: any non-empty raw cell in cols 7-48 that failed to parse would show up here
n_parse_failures = sum(
    parse_number(v) is None for col in RAW_COLS_7_48 for v in raw_const[col] if pd.notna(v)
)
checks_01.append((
    "raw parse failures (cols 7-48)", f"{len(RAW_COLS_7_48)} columns x 299 rows",
    f"{n_parse_failures} failures", n_parse_failures == 0,
))

check_df = pd.DataFrame(checks_01, columns=["check_type", "column", "result", "pass"])
n_fail = int((~check_df["pass"]).sum())
print(f"0.1 summary: {len(check_df)} checks run across {check_df['check_type'].nunique()} check "
      f"types, {n_fail} failed")
print(check_df.groupby("check_type")["pass"].agg(n_checked="count", n_passed="sum"))
if n_fail:
    print("\nFAILING CHECKS:")
    print(check_df.loc[~check_df["pass"]].to_string(index=False))
else:
    print("\nNo failing rows -- every check above passed.")


**Takeaway.** All checks pass cleanly, across all 42 raw columns (cols 7–48) and every column derived from them in `federal_baseline_eng_clean.csv`. No thousands-separator ambiguity exists to trip up decimal-comma parsing (zero cells in this column range contain a literal `.` anywhere), the two `(in 1,000s)` population columns scale by exactly 1000 (max deviation ~1e-11, floating-point noise from the `Decimal`→`float` round-trip, not a real discrepancy) and no other raw column was scaled, every `_pct` column falls inside [0, 100], every population/count column is non-negative, every per-1,000/per-capita column falls in a plausible real-world range (unemployment 2.5–14.8%, not the 300%-style corruption this check was designed to catch), and there are zero unexpected nulls and zero raw parse failures across all 299 constituencies. This is a clean pass, not a manufactured one -- the pipeline documented in `clean_federal_baseline_eng.py` and `README_data_processing.md` is doing exactly what it says.

## 2.1 Building the constituency-level analysis table

`const_agg` (built in Part 1) already carries `constituency_code`/`constituency`/`state` in the same form as `federal_baseline_eng_selected_political.csv` (both ultimately keyed off the same `federal_baseline_eng_selected_political.csv` backbone, so there's no naming mismatch left to fix here — that mismatch was between the raw shapefile fields (`WKR_NR`/`LAND_NR`) and this file, and we already resolved it in Section 1). We merge in every other socioeconomic/political column.

In [ ]:
political

In [ ]:
political_extra = political.drop(columns=["state", "constituency_code"])
analysis_df = const_agg.merge(political_extra, on="constituency", how="left")
assert analysis_df["total_population"].notna().all(), "every constituency should have matched socioeconomic data"

# second racialised rate: over *all* text reviews rather than social-only (racialised_rate above),
# to see whether diluting the denominator with off-topic reviews changes which predictors matter
analysis_df["racialised_rate_of_all"] = analysis_df["racialised_reviews"] / analysis_df["total_text_reviews"]

print(f"analysis_df: {analysis_df.shape[0]} constituencies x {analysis_df.shape[1]} cols")
analysis_df[["constituency", "state", "racialised_rate", "racialised_rate_of_all", "total_population", "second_vote_afd_pct"]].head()


In [ ]:
# Task 2 pipeline check: clean_federal_baseline_eng.py now selects `population_density_per_km2`
# and `area_km2` through to federal_baseline_eng_selected.csv (both were already computed via
# SHORT_COLUMN_NAMES but not previously included in SELECTED_COLUMNS). const_agg was also rebuilt
# since this notebook last ran, replacing the coastline-inclusive `is_border_constituency` with
# the corrected `is_land_border_constituency` (land-neighbour distance <150m only, coastline
# excluded — the old flag wrongly put coastal-but-inland seats like `Bremen II – Bremerhaven` on
# the same footing as a genuine cross-border constituency like `Offenburg`) plus the underlying
# `distance_to_neighbor_country_km`. Confirm all four landed in analysis_df through the existing
# generic merge above, with no unexpected NaNs.
new_pipeline_cols = ["population_density_per_km2", "area_km2",
                      "is_land_border_constituency", "distance_to_neighbor_country_km"]
print("present in analysis_df:", {c: c in analysis_df.columns for c in new_pipeline_cols})
print("NaN counts:")
print(analysis_df[new_pipeline_cols].isna().sum())

berlin_density = analysis_df.loc[analysis_df["state"] == "Berlin", "population_density_per_km2"]
rural_row = analysis_df.loc[analysis_df["population_density_per_km2"].idxmin()]
print(f"\nBerlin density range: {berlin_density.min():.1f}-{berlin_density.max():.1f} inhabitants/km²")
print(f"Least dense constituency: {rural_row['constituency']} ({rural_row['state']}), "
      f"{rural_row['population_density_per_km2']:.1f} inhabitants/km²")
print(f"\nis_land_border_constituency: {analysis_df['is_land_border_constituency'].sum()} / {len(analysis_df)} True "
      f"(land-neighbour only, from Part 1's corrected geometric check)")


## 2.2 A framing caveat, stated once and applied throughout Part 2

Everything below correlates a **constituency-aggregate** rate (racialised reviews / social reviews, computed across all reviewers who happened to review a pool in that constituency) with **constituency-aggregate** socioeconomic/political indicators (population shares, vote shares, etc). This is an **ecological-level analysis**: a finding like "constituencies with a higher AfD vote share tend to have a higher racialised-review rate" is a claim about *places*, not about *individual reviewers* — it does not mean AfD voters wrote the racialised reviews, or that any individual reviewer's characteristics can be inferred from their constituency's aggregate statistics (the ecological fallacy). This framing matters for how any of Part 2's results should be written up for an ICWSM audience: as area-level associations that motivate further (ideally individual- or review-level) investigation, not as claims about who is behind the discourse.

A sharper version of this same concern is whether a "constituency-level" pattern is really a *one-venue* pattern wearing a constituency's name — Part 1 §1.2b checked this directly: racialised rate correlates strongly with how many of a constituency's *pools* are affected (ρ=0.65), not with how much its single most-implicated pool dominates (ρ=0.09, not significant). So the area-level framing below is reasonably earned, not just a formality — high-rate constituencies tend to be broadly, not narrowly, affected.

## 2.3 Deriving comparable rate/per-capita variables

The socioeconomic file mixes raw totals (`total_population`, `foreign_population`, `disposable_income_total_eur`, ...), already-published rates (`unemployment_total`, `second_vote_*_pct`, `turnout_pct`), and derived counts (`employees_total`, `sgb_ii_recipients_total`, ...). Correlating `racialised_rate` (already a rate) against raw population-scaled totals would mostly just reflect constituency size, not composition — so we convert every relevant total into a per-capita or percentage form ourselves, following the same `x / total_population` logic already documented in the socioeconomic README's derivation notes (not inventing a new formula, just extending the same one to the columns we need for correlation that weren't already expressed as a rate).

In [ ]:
pop = analysis_df["total_population"]

analysis_df["foreign_population_pct"] = analysis_df["foreign_population"] / pop * 100
for age_col in ["age_under_18_population", "age_18_24_population", "age_25_34_population",
                "age_35_59_population", "age_60_74_population", "age_75_plus_population"]:
    analysis_df[age_col.replace("_population", "_pct")] = analysis_df[age_col] / pop * 100

analysis_df["disposable_income_eur_per_capita"] = analysis_df["disposable_income_total_eur"] / pop
analysis_df["gdp_eur_per_capita"] = analysis_df["gdp_total_eur"] / pop
analysis_df["employees_pct_of_pop"] = analysis_df["employees_total"] / pop * 100
analysis_df["sgb_ii_recipients_pct"] = analysis_df["sgb_ii_recipients_total"] / pop * 100

PREDICTOR_COLS = [
    "foreign_population_pct", "age_under_18_pct", "age_18_24_pct", "age_25_34_pct",
    "age_60_74_pct", "age_75_plus_pct", "disposable_income_eur_per_capita", "gdp_eur_per_capita",
    "employees_pct_of_pop", "sgb_ii_recipients_pct", "unemployment_total", "turnout_pct",
    "second_vote_spd_pct", "second_vote_cdu_csu_pct", "second_vote_greens_pct",
    "second_vote_afd_pct", "second_vote_left_pct", "second_vote_bsw_pct",
]
print(f"{len(PREDICTOR_COLS)} predictors prepared for correlation analysis")
analysis_df[PREDICTOR_COLS].describe().T[["mean", "std", "min", "max"]]

# offset columns for Part 3's count model (NB2 with a review-volume offset)
analysis_df["log_social_reviews"] = np.log(analysis_df["social_reviews"])
analysis_df["log_total_text_reviews"] = np.log(analysis_df["total_text_reviews"])


## 2.3b Socioeconomic denominator audit and a Berlin/Hamburg/Bremen caveat

Before treating all 18 `PREDICTOR_COLS` as directly comparable in §2.4, two things are worth checking explicitly: whether they really share a common denominator basis, and whether the merge that built `analysis_df` could be silently duplicating rows.

**Denominator basis.** Of the 18 predictors, **10** are population-per-capita variables derived directly above in §2.3 (`x / total_population`, or `x / total_population * 100`): `foreign_population_pct`, the five age-share columns, `disposable_income_eur_per_capita`, `gdp_eur_per_capita`, `employees_pct_of_pop`, and `sgb_ii_recipients_pct`. **7** are election variables denominated by `valid_votes` or `eligible_voters` (`turnout_pct` and the six `second_vote_*_pct` columns) -- not population-based by construction, so not really a methodological trap since nobody would expect otherwise. **`unemployment_total` is the one that matters**: per `README_data_processing.md`'s explicit calibration note ("Unemployment rates are kept as rates... the proper denominator is the labor force, not total population"), it is a published labor-force-denominated rate, not something derived from `total_population` the way the 10 population-share predictors are. It sits directly alongside `sgb_ii_recipients_pct`/`foreign_population_pct` in §2.4's ranking (ρ=0.43, the third-strongest predictor) -- a reader could reasonably assume all three are on the same population basis. They are not, and that should be flagged whenever `unemployment_total` is cited alongside the population-share predictors.

**Merge integrity.** `constituency_code` -- the actual join key -- is unique in both `political` and `const_agg` before the merge, and `analysis_df` has exactly as many rows after it as before: a clean 1:1 join, no row duplication.

In [ ]:
# denominator audit: split PREDICTOR_COLS into population-per-capita vs. election-denominated
# vs. the one labor-force-denominated exception
population_denominated = [
    "foreign_population_pct", "age_under_18_pct", "age_18_24_pct", "age_25_34_pct",
    "age_60_74_pct", "age_75_plus_pct", "disposable_income_eur_per_capita",
    "gdp_eur_per_capita", "employees_pct_of_pop", "sgb_ii_recipients_pct",
]
election_denominated = [
    "turnout_pct", "second_vote_spd_pct", "second_vote_cdu_csu_pct", "second_vote_greens_pct",
    "second_vote_afd_pct", "second_vote_left_pct", "second_vote_bsw_pct",
]
labor_force_denominated = ["unemployment_total"]
assert set(population_denominated + election_denominated + labor_force_denominated) == set(PREDICTOR_COLS)
print(f"{len(population_denominated)} population-per-capita predictors")
print(f"{len(election_denominated)} election-denominated predictors (valid_votes / eligible_voters)")
print(f"{len(labor_force_denominated)} labor-force-denominated predictor: {labor_force_denominated}")

# merge integrity: is constituency_code -- the actual join key -- unique in both source tables,
# and did the merge inflate the row count?
print(f"\npolitical: {political['constituency_code'].nunique()} unique constituency_code / {len(political)} rows")
print(f"const_agg: {const_agg['constituency_code'].nunique()} unique constituency_code / {len(const_agg)} rows")
print(f"analysis_df: {len(analysis_df)} rows after merge (no inflation)")

n_dup_pop_values = analysis_df["total_population"].duplicated().sum()
print(f"\ntotal_population values that happen to match another constituency's: {n_dup_pop_values} "
      f"(coincidence, not a duplication bug -- see takeaway below)")

In [ ]:
# which columns are actually just a shared state-wide rate in Berlin/Hamburg/Bremen, and do any
# of the predictors we actually use in §2.4 inherit that?
repeated_report = pd.read_csv(SOCIO_DIR / "state_repeated_columns_report.csv")
repeated_report["repeated_states_list"] = repeated_report["repeated_states"].fillna("").apply(
    lambda s: [x.strip() for x in s.split(";") if x.strip()]
)
bhb_repeated = repeated_report[repeated_report["repeated_states_list"].apply(
    lambda states: any(s in states for s in ["Berlin", "Hamburg", "Bremen"])
)]
print(f"{len(bhb_repeated)} columns flagged as repeated within Berlin, Hamburg, and/or Bremen:")
print(bhb_repeated[["column", "repeated_states"]].to_string(index=False))

# of those, which feed directly into this notebook's §2.3 predictor derivation?
predictor_ingredient_cols = [
    "age_under_18_pct", "age_18_24_pct", "age_25_34_pct", "age_35_59_pct", "age_60_74_pct", "age_75_plus_pct",
    "disposable_income_eur_per_inhabitant", "gdp_eur_per_inhabitant",
    "sgb_ii_recipients_per_1k", "employees_per_1k", "unemployment_total",
]
predictor_relevant_repeated = bhb_repeated[bhb_repeated["column"].isin(predictor_ingredient_cols)]
print(f"\nof those, {len(predictor_relevant_repeated)} feed directly into a §2.3 predictor:")
print(predictor_relevant_repeated[["column", "repeated_states"]].to_string(index=False))

# spot-check: are these ACTUALLY identical within a state once re-derived as %, or just close?
berlin_check = analysis_df.loc[analysis_df["state"] == "Berlin",
    ["constituency", "sgb_ii_recipients_pct", "disposable_income_eur_per_capita", "gdp_eur_per_capita"]]
print(f"\nBerlin's {len(berlin_check)} constituencies -- unique values of sgb_ii_recipients_pct / "
      f"disposable_income_eur_per_capita / gdp_eur_per_capita: "
      f"{berlin_check['sgb_ii_recipients_pct'].nunique()}, {berlin_check['disposable_income_eur_per_capita'].nunique()}, "
      f"{berlin_check['gdp_eur_per_capita'].nunique()}")

bhb_mask = analysis_df["state"].isin(["Berlin", "Hamburg", "Bremen"])
print(f"\nconstituencies in Berlin/Hamburg/Bremen: {bhb_mask.sum()} / {len(analysis_df)}")
print(analysis_df.loc[bhb_mask, "state"].value_counts())

**Takeaway.** Two real findings and one non-issue. (1) **`unemployment_total` is a genuine denominator mismatch** -- a labor-force rate sitting next to 10 population-share predictors it's implicitly compared against in §2.4's ranking; that comparison should be read with this in mind. (2) **The merge is clean** -- `constituency_code` is unique in every source table and `analysis_df`'s row count doesn't inflate after the join; the 27 `total_population` value collisions are a rounding coincidence between similarly-sized constituencies (Bundestag districts are deliberately population-balanced), not a data-quality problem. (3) **The Berlin/Hamburg caveat is real, and sharper than a 'similar values' concern for two of the three spot-checked columns**: 11 predictor-ingredient columns are flagged as state-wide constants in Berlin and/or Hamburg (the six age-share columns, `disposable_income_eur_per_inhabitant`, `gdp_eur_per_inhabitant`, `sgb_ii_recipients_per_1k`, `employees_per_1k`, and `unemployment_total`). Because `total_population` cancels out algebraically in the per-capita derivation, `disposable_income_eur_per_capita` and `gdp_eur_per_capita` come back out **bit-for-bit identical** across Berlin's 12 constituencies (1 unique value each, confirmed above) -- these two are pure linear reconstructions of a single city-wide EUR-per-inhabitant figure, with no independent constituency-level information at all. `sgb_ii_recipients_pct` is *not* bit-identical (12 distinct values, confirmed above) but is practically constant, varying only in the 4th decimal place (~12.8199-12.8202 across all 12 Berlin constituencies) -- the intermediate step (`sgb_ii_recipients_total`, rounded to a whole number per `README_data_processing.md`) breaks exact algebraic cancellation, but the underlying value is still overwhelmingly the shared Berlin-wide rate re-scaled by local population, not an independently observed constituency-level measurement. Bremen's 2 constituencies, by contrast, don't share any of the columns that actually feed §2.4's predictors (only `craft_enterprises_per_1k`, `employees_agriculture_pct`, and `municipalities` are Bremen-repeated, none of which are used here) -- so this is really an 18-constituency (Berlin+Hamburg) issue for the predictors in play, not a full Berlin/Hamburg/Bremen one, even though the exclusion check below drops all 20 city-state constituencies together for a clean, consistent comparison. This means a chunk of `analysis_df`'s deprivation/demographic predictors aren't independently observed at the constituency level for those 18 places -- they're one shared city rate, re-scaled by local population. §2.4d below re-runs §2.4/§2.4b excluding all 20 Berlin/Hamburg/Bremen constituencies to check whether this actually changes anything (it needs `run_correlations`/`partial_spearman`, defined below in §2.4/§2.4b, so the robustness re-computation itself is deferred to right after §2.4c rather than run here).

## 0.2 Is the derivation actually making a difference?

§2.3 converts every population-denominated `PREDICTOR_COLS` variable from a raw total into a per-capita/percentage form. But most of those raw totals are themselves *already* a derivation: `clean_federal_baseline_eng.py` took an originally-published rate or per-1,000 figure (e.g. `age_under_18_pct`, `foreign_population_pct`, `disposable_income_eur_per_inhabitant`, `employees_per_1k`, `sgb_ii_recipients_per_1k`), multiplied it back out into a rounded whole-number total, and only kept that total in `federal_baseline_eng_selected_political.csv` (the file this notebook actually loads) -- so §2.3's per-capita/percentage predictors are a **round trip**: published rate → derived total (rounded) → re-derived rate. This checks whether that round trip actually preserves the originally-published rate, or introduces meaningful drift, and then folds in §2.3b's Berlin/Hamburg finding as the sharpest case where the derivation doesn't add real constituency-level information at all.

In [ ]:
# round-trip fidelity: published rate -> derived total (rounded, per clean_federal_baseline_eng.py)
# -> re-derived rate (as this notebook's own §2.3 does). Uses federal_baseline_eng_clean.csv
# (loaded above in §0.1 as clean_const), which -- unlike the selected file §2.1-§2.3 actually
# load -- still carries the originally-published rate columns alongside the derived totals, so both
# ends of the round trip are available to compare directly.
pop_full = clean_const["total_population"].astype(float)

ROUND_TRIP_CHECKS = [
    ("age_under_18_pct", "age_under_18_population", lambda cnt: cnt / pop_full * 100),
    ("age_18_24_pct", "age_18_24_population", lambda cnt: cnt / pop_full * 100),
    ("age_25_34_pct", "age_25_34_population", lambda cnt: cnt / pop_full * 100),
    ("age_60_74_pct", "age_60_74_population", lambda cnt: cnt / pop_full * 100),
    ("age_75_plus_pct", "age_75_plus_population", lambda cnt: cnt / pop_full * 100),
    ("foreign_population_pct", "foreign_population", lambda cnt: cnt / pop_full * 100),
    ("disposable_income_eur_per_inhabitant", "disposable_income_total_eur", lambda cnt: cnt / pop_full),
    ("gdp_eur_per_inhabitant", "gdp_total_eur", lambda cnt: cnt / pop_full),
    ("employees_per_1k", "employees_total", lambda cnt: cnt / pop_full * 1000),
    ("sgb_ii_recipients_per_1k", "sgb_ii_recipients_total", lambda cnt: cnt / pop_full * 1000),
]

roundtrip_rows = []
for orig_col, count_col, reconstruct in ROUND_TRIP_CHECKS:
    orig = clean_const[orig_col].astype(float)
    reconstructed = reconstruct(clean_const[count_col].astype(float))
    diff = (orig - reconstructed).abs()
    worst = diff.idxmax()
    roundtrip_rows.append({
        "published_rate": orig_col, "via_derived_total": count_col,
        "max_abs_drift": diff.max(), "n_over_0.05": int((diff > 0.05).sum()),
        "worst_constituency": clean_const.loc[worst, "constituency"],
    })
roundtrip_df = pd.DataFrame(roundtrip_rows).sort_values("max_abs_drift", ascending=False)
print("Round-trip drift (published rate vs. rate reconstructed from the rounded derived total):")
roundtrip_df


In [ ]:
# fold in §2.3b's finding: for the predictor-ingredient columns already flagged as state-wide
# constant in Berlin and/or Hamburg, does the round trip above still "pass" even though there's no
# real constituency-level variation being preserved in the first place?
print(f"§2.3b flagged {len(predictor_relevant_repeated)} predictor-ingredient columns as "
      f"state-wide constant in Berlin/Hamburg: {predictor_relevant_repeated['column'].tolist()}")
print(f"\nBerlin's 12 constituencies -- unique reconstructed values for the same three columns "
      f"checked in §2.3b: sgb_ii_recipients_pct={berlin_check['sgb_ii_recipients_pct'].nunique()}, "
      f"disposable_income_eur_per_capita={berlin_check['disposable_income_eur_per_capita'].nunique()}, "
      f"gdp_eur_per_capita={berlin_check['gdp_eur_per_capita'].nunique()} (of 12 possible)")
print("\nThe round-trip check above and this state-constancy check catch two different failure "
      "modes: round-trip fidelity asks whether converting a rate to a total and back distorts the "
      "*value*; state-constancy asks whether there was ever independent constituency-level "
      "information to distort in the first place. A perfectly state-wide-constant rate round-trips "
      "almost exactly (there's nothing for rounding to disturb), which is exactly what both checks "
      "show for Berlin/Hamburg -- passing the value-fidelity check above does not contradict "
      "§2.3b's finding.")


**Takeaway -- Part 0 bottom line.** No new problems, and one confirmed-benign exception. Round-trip fidelity is essentially exact for 9 of 10 checked predictors (max drift on the order of 0.01 percentage points / EUR / per-1,000 units -- rounding noise from `clean_federal_baseline_eng.py`'s whole-number rounding, not a distortion worth worrying about). The one exception, `foreign_population_pct`, drifts by up to ~0.075 percentage points (roughly 30 of 299 constituencies exceed a 0.05pp threshold) -- but this is not a corruption or a rounding artifact of *this* round trip: `foreign_population` is deliberately *not* derived from `foreign_population_pct` (the pipeline instead subtracts two independently-rounded actual counts, per `README_data_processing.md`, specifically because the published percentage is only accurate to 1 decimal place), so this is the data's own reporting precision showing through, not a bug. Folding in §2.3b: the round-trip check and the state-constancy finding are not in tension -- a shared state-wide rate re-scaled by local population round-trips almost perfectly (nothing for rounding to disturb) while still carrying zero independent constituency-level information for the 18 Berlin/Hamburg constituencies already flagged there. **Bottom line: this derivation adds legitimate constituency-level information everywhere except those 18 (of 299) constituencies, where it repackages a shared city rate rather than measuring anything new -- exactly what §2.3b already found, now confirmed from the value-fidelity side rather than just the state-constancy side.** §2.4d later re-runs the correlations excluding all 20 Berlin/Hamburg/Bremen constituencies to check whether this actually changes the substantive results.

# Correlations, relationships, and spatial analysis

Part 0 above established that the raw data survived cleaning intact and that this notebook's own rate/per-capita derivations add real information except for the already-flagged Berlin/Hamburg constituencies. Everything from here on (§2.4 onward) builds on that: Spearman correlations against the socioeconomic/political predictors, confound checks, group comparisons, spatial autocorrelation, and political cross-tabs.

## 2.4 Correlations: racialised rate vs. socioeconomic/political predictors

**Method choices:**
- **Spearman**, not Pearson: several predictors (e.g. `gdp_eur_per_capita`, `sgb_ii_recipients_pct`) are right-skewed, and `racialised_rate` itself is a small proportion bounded at 0 — Spearman only assumes a monotonic relationship, which is more defensible here than assuming linearity.
- **Benjamini-Hochberg FDR correction** across the 18 simultaneous predictor tests, rather than treating each p-value in isolation — with this many tests, some would clear p<.05 by chance alone.
- **Bootstrap CI on rho** (constituency-level resampling, 2,000 draws) reported alongside the point estimate, since a correlation computed over 299 units deserves an uncertainty range, not just a point estimate + p-value.
- **Robustness check**: every correlation is recomputed restricted to the 247 constituencies that clear Part 1's reliability bar (`racialised_reviews >= 10`, tightened from the earlier `>= 5` bar), so we can see whether any result is being driven by the 52 thin constituencies.
- **Two rate definitions, compared side by side**: `racialised_rate` (racialised / social reviews, Part 1's chosen definition) alongside `racialised_rate_of_all` (racialised / all text reviews), to see whether diluting the denominator with reviews never in scope for racialised-discourse coding changes which predictors matter or how strongly.


In [ ]:
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(seed=42)

def spearman_with_bootstrap_ci(x, y, n_boot=2000):
    rho, p = stats.spearmanr(x, y)
    n = len(x)
    boot_rhos = np.empty(n_boot)
    x_arr, y_arr = np.asarray(x), np.asarray(y)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_rhos[i] = stats.spearmanr(x_arr[idx], y_arr[idx])[0]
    ci_lo, ci_hi = np.nanpercentile(boot_rhos, [2.5, 97.5])
    return rho, p, ci_lo, ci_hi

def run_correlations(df, target_col="racialised_rate"):
    rows = []
    for col in PREDICTOR_COLS:
        rho, p, ci_lo, ci_hi = spearman_with_bootstrap_ci(df[col], df[target_col])
        rows.append({"predictor": col, "rho": rho, "p": p, "ci_lo": ci_lo, "ci_hi": ci_hi})
    out = pd.DataFrame(rows)
    out["p_fdr"] = multipletests(out["p"], method="fdr_bh")[1]
    out["significant_fdr"] = out["p_fdr"] < 0.05
    return out.sort_values("rho", key=np.abs, ascending=False).reset_index(drop=True)

corr_all = run_correlations(analysis_df)
corr_reliable = run_correlations(analysis_df[analysis_df["reliable_n"]])

print("All 299 constituencies:")
print(corr_all.to_string(index=False))

corr_all_rate2 = run_correlations(analysis_df, target_col="racialised_rate_of_all")

rate_comparison = corr_all[["predictor", "rho", "p_fdr", "significant_fdr"]].merge(
    corr_all_rate2[["predictor", "rho", "p_fdr", "significant_fdr"]],
    on="predictor", suffixes=("_social", "_all"),
)
rate_comparison["rho_diff"] = rate_comparison["rho_all"] - rate_comparison["rho_social"]
print("\nBoth rate definitions, all 299 constituencies (racialised_rate = /social, racialised_rate_of_all = /all text reviews):")
print(rate_comparison.sort_values("rho_diff", key=np.abs, ascending=False).to_string(index=False))


In [ ]:
# robustness: compare rho with vs without the constituencies below §1.2's racialised/social-review
# reliability bar (not a fixed count — depends on Part 1's reliable_n column)
n_reliable_constituencies = analysis_df["reliable_n"].sum()
robustness = corr_all[["predictor", "rho"]].merge(
    corr_reliable[["predictor", "rho"]], on="predictor", suffixes=("_all", "_reliable_only")
)
robustness["rho_shift"] = robustness["rho_reliable_only"] - robustness["rho_all"]
print(f"Robustness check (rho using all 299 vs. the {n_reliable_constituencies} 'reliable' constituencies):")
print(robustness.sort_values("rho_shift", key=np.abs, ascending=False).to_string(index=False))

# forest plot, sorted by |rho|, using the full-sample (all 299) results
plot_df = corr_all.sort_values("rho")
fig, ax = plt.subplots(figsize=(8, 7))
colors = np.where(plot_df["significant_fdr"], "#C44E52", "#8C8C8C")
ax.errorbar(
    plot_df["rho"], range(len(plot_df)),
    xerr=[plot_df["rho"] - plot_df["ci_lo"], plot_df["ci_hi"] - plot_df["rho"]],
    fmt="o", color="black", ecolor=list(colors), markersize=4,
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df["predictor"])
ax.set_xlabel("Spearman rho vs. racialised_rate (bootstrap 95% CI)")
ax.set_title("Correlations with racialised rate\n(red = significant after Benjamini-Hochberg FDR correction)")
plt.tight_layout()
plt.show()


In [ ]:
# the AfD-vote-share result runs counter to a naive prior (higher AfD share -> higher racialised rate);
# before over-interpreting, check whether predictors are themselves intercorrelated in a way that could
# explain this (univariate correlations can't separate confounded predictors from each other)
key_predictor_corr = analysis_df[["second_vote_afd_pct", "foreign_population_pct", "sgb_ii_recipients_pct", "unemployment_total"]].corr(method="spearman")
print("Spearman correlations among the predictors themselves:")
print(key_predictor_corr.round(3))


**Takeaway.** This is the most substantial result in the notebook: **17 of 18 predictors are significant after FDR correction** (only `age_under_18_pct` isn't), and the robustness check confirms none of it is being driven by the 52 constituencies below §1.2's racialised/social-review reliability bar (Part 1's bar was tightened from `racialised_reviews >= 5` to `>= 10` in the most recent refresh, so this is a larger held-out set than in an earlier pass of this notebook) -- rho shifts are larger than before (up to 0.091 for `second_vote_left_pct`) but every predictor keeps the same sign and none changes FDR-significance status. The strongest, most defensible associations are **deprivation and composition variables**: `sgb_ii_recipients_pct` (ρ=0.47), `foreign_population_pct` (ρ=0.46), and `unemployment_total` (ρ=0.43) — constituencies with more social-benefit dependency, more residents with a foreign-population share, and higher unemployment show a higher racialised-review rate. Age structure follows a coherent pattern too: younger-skewing constituencies (`age_25_34_pct` +0.33, `age_18_24_pct` +0.24) show higher rates, older ones (`age_60_74_pct` −0.34, `age_75_plus_pct` −0.22) lower.

The political pattern is the one that most needs careful handling: `second_vote_left_pct` (+0.34), `second_vote_spd_pct` (+0.28), and `second_vote_greens_pct` (+0.24) are positive, while `second_vote_cdu_csu_pct` (−0.27) and, notably, **`second_vote_afd_pct` (−0.20) are negative** — the opposite direction of the naive prior that higher AfD support would go with more racialised discourse. Checking the predictors against each other shows why: `second_vote_afd_pct` is strongly negatively correlated with `foreign_population_pct` itself (ρ=−0.54) — AfD support in this data is concentrated in lower-foreign-population constituencies, and `sgb_ii_recipients_pct`/`unemployment_total` are themselves almost collinear (ρ=0.96, unsurprising — they're two views of the same underlying deprivation). **A univariate correlation matrix cannot separate these confounded predictors from each other or say which one is doing the explanatory work** — this is exactly the ecological-fallacy and confounding risk flagged in §2.2, and exactly why Part 2 stops at correlation rather than attempting a multivariate model in this pass: any single-predictor story (e.g. "AfD support is associated with less racialised discourse") would be misleading without controlling for the others, which needs the multivariate model in Part 3 (§3.3) below, not a univariate correlation matrix.

**Rate-definition comparison.** Swapping the denominator from `social_reviews` to `total_text_reviews` (`racialised_rate_of_all`) barely moves anything: all 17 FDR-significant predictors stay significant under both definitions with the same sign, `age_under_18_pct` stays the one non-significant predictor under both, and every |rho_diff| is small (max 0.033, for `unemployment_total` and `age_18_24_pct`) against rho magnitudes of 0.05-0.50. Practically, the two rates are redundant for this correlation pass -- Part 1's `/social` choice (excluding reviews never in scope for racialised-discourse coding, see its rationale) isn't costing anything here, and doesn't need to be revisited before moving to the regression stage.


## 2.4b Is `racialised_rate` itself confounded by population size or venue count?

Every predictor above was normalized to a per-capita/percentage form (§2.3) specifically to avoid a constituency-size artifact on the *predictor* side. But we haven't yet checked the *outcome* side: `racialised_rate` is racialised/social reviews, which controls for review volume by construction, but it could still be mechanically related to how many pools a constituency has (`pool_count`) or how many people live there (`total_population`) — e.g. if bigger, denser constituencies happen to generate systematically different rates for reasons that have nothing to do with deprivation or demographics. If that's true, the §2.4 correlations could be partly an artifact of constituency size rather than composition.

We test this directly (Spearman, same method as §2.4), and then — since the honest answer turns out to require it — recompute the strongest §2.4 predictors as **partial correlations controlling for whichever size variable turns out to matter**, using rank-residualization (regress the ranks of each variable on the ranks of the control variable via OLS, then Spearman-correlate the residuals — the standard nonparametric equivalent of a partial correlation, since `pingouin` isn't installed in this environment).

In [ ]:
SIZE_VARS = ["pool_count", "total_text_reviews", "total_population"]

rows = []
for col in SIZE_VARS:
    rho, p, ci_lo, ci_hi = spearman_with_bootstrap_ci(analysis_df[col], analysis_df["racialised_rate"])
    rows.append({"size_variable": col, "rho": rho, "p": p, "ci_lo": ci_lo, "ci_hi": ci_hi})
size_check = pd.DataFrame(rows)
size_check["p_fdr"] = multipletests(size_check["p"], method="fdr_bh")[1]
print("racialised_rate vs. constituency-size variables:")
print(size_check.round(4).to_string(index=False))


In [ ]:
import statsmodels.api as sm
from scipy.stats import rankdata

def partial_spearman(x, y, control):
    x, y, control = np.asarray(x, dtype=float), np.asarray(y, dtype=float), np.asarray(control, dtype=float)
    rx, ry, rz = rankdata(x), rankdata(y), rankdata(control)
    Z = sm.add_constant(rz)
    rx_resid = sm.OLS(rx, Z).fit().resid
    ry_resid = sm.OLS(ry, Z).fit().resid
    return stats.pearsonr(rx_resid, ry_resid)

# how confounded are the §2.4 predictors themselves with pool_count (venue density)?
predictor_vs_poolcount = pd.Series(
    {col: stats.spearmanr(analysis_df[col], analysis_df["pool_count"])[0] for col in PREDICTOR_COLS},
    name="rho_vs_pool_count",
).sort_values(key=np.abs, ascending=False)
print("How correlated is each §2.4 predictor with pool_count (venue density)?")
print(predictor_vs_poolcount.round(3).to_string())

# partial correlations, controlling for pool_count, for the predictors that were significant in §2.4
partial_rows = []
for col in PREDICTOR_COLS:
    r_partial, p_partial = partial_spearman(analysis_df[col], analysis_df["racialised_rate"], analysis_df["pool_count"])
    original_rho = corr_all.loc[corr_all["predictor"] == col, "rho"].iloc[0]
    partial_rows.append({"predictor": col, "rho_original": original_rho, "rho_partial_ctrl_poolcount": r_partial,
                          "p_partial": p_partial, "shift": r_partial - original_rho})
partial_df = pd.DataFrame(partial_rows).sort_values("rho_original", key=np.abs, ascending=False)
partial_df["p_partial_fdr"] = multipletests(partial_df["p_partial"], method="fdr_bh")[1]
partial_df["still_significant_fdr"] = partial_df["p_partial_fdr"] < 0.05
print("\nOriginal vs. pool_count-controlled partial Spearman correlations:")
print(partial_df.round(3).to_string(index=False))


**Takeaway — this is an important correction to §2.4, not a footnote.** `racialised_rate` is substantially confounded by `pool_count` (ρ=−0.49, p<0.0001) — constituencies with more pools have systematically *lower* rates — and more weakly by `total_population` (ρ=0.27) and `total_text_reviews` (ρ=−0.13). Worse, **almost every §2.4 predictor is itself strongly correlated with `pool_count`** (|ρ| = 0.32-0.57 for most of them) — deprivation, age structure, and vote share are all entangled with venue density in this data, likely because pool_count is a proxy for urban/rural character (dense cities have fewer, larger, heavily-reviewed pools; spread-out rural areas have more, smaller ones).

Controlling for `pool_count` via partial Spearman correlation, **only 6 of 18 predictors remain significant after FDR correction**: `sgb_ii_recipients_pct` (ρ 0.47→0.26), `foreign_population_pct` (0.46→0.28), `unemployment_total` (0.43→0.23), `turnout_pct` (−0.27→−0.26, barely moved), `second_vote_spd_pct` (0.28→0.14), and `second_vote_bsw_pct` (0.19→0.15). Everything else loses significance once venue density is accounted for — including `second_vote_left_pct`, the age-structure variables, `second_vote_greens_pct`, `gdp_eur_per_capita`, and `employees_pct_of_pop`.

**`second_vote_afd_pct` is the clearest casualty**: its partial correlation collapses from ρ=−0.20 to ρ=+0.03 (p=0.58, nowhere near significant) once `pool_count` is controlled — even more decisively confounded than the §2.4 finding against `foreign_population_pct` alone suggested. This *strengthens* rather than changes that section's conclusion: the AfD association was never independent, and controlling for venue density kills it more thoroughly than controlling for foreign population share did.

**Net effect**: the deprivation cluster (`sgb_ii_recipients_pct`, `foreign_population_pct`, `unemployment_total`) and, more surprisingly, `turnout_pct` are the only predictors that look genuinely robust to this specific confound — and even they are meaningfully attenuated, not just "confirmed." This is exactly the kind of result that argues for a proper multivariate model (with `pool_count` as a covariate, not just a nuisance to partial out one at a time) rather than reading any of §2.4's univariate correlations at face value — reinforcing, with much more evidence now, why Part 3's multivariate model below controls for `pool_count` directly rather than this section's one-at-a-time partial correlations.

## 2.4c Robustness check: are the correlations driven by venue-concentration outliers?

§1.2b-§1.2c (notebook 01) found that most constituencies' racialised rate is broadly spread across their pools, and their "dominant" pool is usually just a big, popular venue — but flagged **19 constituencies where the dominant pool is a small venue with a genuinely extreme rate** (`Lübeck` and `Mansfeld` most sharply, both at 23.8%). Those 19 are exactly the candidates for a real hole in the analysis: if the §2.4/§2.4b correlations depend on them, the "deprivation cluster" story could partly be an artifact of a handful of small, outlier-driven venues rather than a genuine area-level pattern.

We rerun the §2.4 correlations excluding those 19 constituencies (analogous to the existing low-`N` robustness check, just against a different, substantively motivated exclusion set) and compare rho shifts directly.

In [ ]:
n_flagged = analysis_df["dominant_pool_is_small_high_rate"].sum()
print(f"constituencies flagged as venue-concentration outliers: {n_flagged} / {len(analysis_df)}")

excl_df = analysis_df[~analysis_df["dominant_pool_is_small_high_rate"]]

# univariate correlations (§2.4), with vs. without the 19 flagged constituencies
corr_excl_concentration = run_correlations(excl_df)
univariate_check = corr_all[["predictor", "rho"]].merge(
    corr_excl_concentration[["predictor", "rho"]], on="predictor", suffixes=("_all", "_excl_concentrated")
)
univariate_check["shift"] = univariate_check["rho_excl_concentrated"] - univariate_check["rho_all"]
print("\nUnivariate rho: all 299 vs. excluding the 19 venue-concentration outliers:")
print(univariate_check.sort_values("shift", key=np.abs, ascending=False).to_string(index=False))

# partial correlations (§2.4b, controlling for pool_count), same comparison, for the 6 predictors
# that survived that check
survivors = partial_df.loc[partial_df["still_significant_fdr"], "predictor"].tolist()
partial_rows = []
for col in survivors:
    r_partial, p_partial = partial_spearman(excl_df[col], excl_df["racialised_rate"], excl_df["pool_count"])
    original = partial_df.loc[partial_df["predictor"] == col, "rho_partial_ctrl_poolcount"].iloc[0]
    partial_rows.append({"predictor": col, "partial_rho_all": original, "partial_rho_excl_concentrated": r_partial,
                          "shift": r_partial - original, "p_excl_concentrated": p_partial})
print("\nSame check on the 6 pool_count-controlled §2.4b survivors:")
print(pd.DataFrame(partial_rows).round(3).to_string(index=False))


**Takeaway.** This hole doesn't hold up — the correlations are not being driven by the venue-concentration outliers. Excluding the 19 flagged constituencies shifts every univariate rho by at most 0.019 (`age_18_24_pct`), and the 6 predictors that survived §2.4b's pool-count control shift by at most 0.022 (`sgb_ii_recipients_pct`) while staying significant at essentially the same p-values. Nothing changes sign, nothing crosses a significance threshold either direction.

Put together with §1.2b-§1.2c: constituency-level racialised rate is (a) mostly driven by broad-based patterns across many pools rather than single venues, and (b) even in the minority of constituencies where one small venue really is an outlier, removing those specific constituencies doesn't move the Part 2 findings. That's a real answer to "could this all just be a few outlier venues" — no, not by this check. It doesn't make the ecological-fallacy caveat in §2.2 go away (these are still area-level associations, not claims about individuals), but it does rule out the specific worry that they're an artifact of a handful of unusual pools.

## 2.4d Robustness check: Berlin/Hamburg/Bremen city-state constituencies

§2.3b flagged that several §2.3 predictors are state-wide constants (not independently observed constituency-level measurements) for 18 of Berlin's and Hamburg's constituencies, while Bremen's 2 don't share any of the columns that actually feed §2.4's predictors. We nonetheless drop all 20 Berlin/Hamburg/Bremen constituencies together here, matching the city-state grouping used throughout this notebook and in the audit report, and re-run §2.4's raw correlations and §2.4b's `pool_count`-controlled partial correlations excluding them -- using the same rho-shift comparison as §2.4c's venue-concentration check, against the same ~0.02 'no meaningful change' benchmark that check established.

In [ ]:
bhb_mask = analysis_df["state"].isin(["Berlin", "Hamburg", "Bremen"])
print(f"excluding {bhb_mask.sum()} Berlin/Hamburg/Bremen constituencies "
      f"({analysis_df.loc[bhb_mask, 'state'].value_counts().to_dict()})")

excl_bhb_df = analysis_df[~bhb_mask]

corr_excl_bhb = run_correlations(excl_bhb_df)
bhb_check = corr_all[["predictor", "rho", "significant_fdr"]].merge(
    corr_excl_bhb[["predictor", "rho", "significant_fdr"]], on="predictor", suffixes=("_all", "_excl_bhb")
)
bhb_check["shift"] = bhb_check["rho_excl_bhb"] - bhb_check["rho_all"]
bhb_check["sig_flip"] = bhb_check["significant_fdr_all"] != bhb_check["significant_fdr_excl_bhb"]
print(f"\nUnivariate rho: all 299 vs. excluding the {bhb_mask.sum()} Berlin/Hamburg/Bremen constituencies:")
print(bhb_check.sort_values("shift", key=np.abs, ascending=False).round(4).to_string(index=False))
print(f"\nmax |shift| = {bhb_check['shift'].abs().max():.4f}, any sign flips: "
      f"{((corr_all.set_index('predictor')['rho'] * corr_excl_bhb.set_index('predictor')['rho']) < 0).any()}, "
      f"significance flips: {bhb_check.loc[bhb_check['sig_flip'], 'predictor'].tolist()}")

In [ ]:
# same check on the 6 pool_count-controlled §2.4b survivors
partial_bhb_rows = []
for col in survivors:
    r_partial, p_partial = partial_spearman(excl_bhb_df[col], excl_bhb_df["racialised_rate"], excl_bhb_df["pool_count"])
    original = partial_df.loc[partial_df["predictor"] == col, "rho_partial_ctrl_poolcount"].iloc[0]
    partial_bhb_rows.append({"predictor": col, "partial_rho_all": original, "partial_rho_excl_bhb": r_partial,
                             "shift": r_partial - original, "p_excl_bhb": p_partial})
partial_bhb_df = pd.DataFrame(partial_bhb_rows)
print("pool_count-controlled partial rho, all 299 vs. excluding Berlin/Hamburg/Bremen:")
print(partial_bhb_df.round(4).to_string(index=False))
print(f"\nmax |shift| = {partial_bhb_df['shift'].abs().max():.4f}")

**Takeaway.** Unlike the venue-concentration check in §2.4c, this one does move the needle somewhat, though not enough to change the notebook's substantive conclusions. Excluding the 20 Berlin/Hamburg/Bremen constituencies shifts univariate rho by more than the ~0.02 'no meaningful change' benchmark for **most** predictors (up to ~0.05 for `employees_pct_of_pop`; `second_vote_afd_pct` and `unemployment_total` both shift by ~0.04-0.05 as well) -- the 20 city-state constituencies pull their correlated predictors in the same direction those predictors already pointed, since Berlin/Hamburg/Bremen run high on both `racialised_rate` (mean ~2.5% vs. ~1.5% elsewhere) and several of the deprivation/composition predictors, so dropping them shrinks (not reverses) most associations. **Nothing flips sign.** **One predictor does cross the FDR-significance line**: `disposable_income_eur_per_capita` stops clearing FDR-significance once Berlin/Hamburg/Bremen are excluded -- notable given it's exactly one of the columns flagged in §2.3b as a Berlin/Hamburg state-wide constant. The `pool_count`-controlled partial correlations for the 6 §2.4b survivors are considerably more stable (shifts well below the 0.02 benchmark, no significance changes) -- once venue density is already controlled for, excluding the city-states barely moves anything further. **Net read**: the deprivation-cluster story (`sgb_ii_recipients_pct`, `foreign_population_pct`, `unemployment_total`) survives excluding Berlin/Hamburg/Bremen and stays FDR-significant both raw and pool_count-partialled, but the raw §2.4 rankings are somewhat inflated by these 20 constituencies (exactly what §2.3b's 'shared city-wide rate' caveat predicts), and one weaker predictor (`disposable_income_eur_per_capita`) doesn't survive without them.

In [ ]:
# do 1 and 2 together: excluding Berlin/Hamburg/Bremen (above) is one way to handle the 10
# state-constant predictors flagged in §2.3b; the other is to just not use them. Report both
# side by side rather than picking one -- does the deprivation-cluster story still hold using
# *only* the predictors with zero state-constancy contamination?
CONTAMINATED_PREDICTORS = [c for c in population_denominated if c != "foreign_population_pct"] + labor_force_denominated
clean_predictors = [c for c in PREDICTOR_COLS if c not in CONTAMINATED_PREDICTORS]
print(f"{len(CONTAMINATED_PREDICTORS)} contaminated predictors (state-wide constant in Berlin/Hamburg): "
      f"{CONTAMINATED_PREDICTORS}")
print(f"{len(clean_predictors)} clean predictors (never flagged): {clean_predictors}\n")

print("All 299, clean predictors only:")
print(corr_all[corr_all["predictor"].isin(clean_predictors)].to_string(index=False))
print("\nExcluding Berlin/Hamburg/Bremen, clean predictors only:")
print(corr_excl_bhb[corr_excl_bhb["predictor"].isin(clean_predictors)].to_string(index=False))


**Takeaway.** The political-lean pattern from §2.4 doesn't rest on any of the 10 flagged columns at all — every political predictor is clean by construction (election-denominated, not population-share-denominated), so `second_vote_left_pct` (+0.34), `second_vote_spd_pct` (+0.28), `second_vote_greens_pct` (+0.24) positive and `second_vote_cdu_csu_pct` (−0.27), `second_vote_afd_pct` (−0.20) negative all stand exactly as reported in §2.4, with or without Berlin/Hamburg/Bremen. What the 10 contaminated columns take with them is most of the *deprivation* story's specific texture: `sgb_ii_recipients_pct`, `unemployment_total`, `disposable_income_eur_per_capita`, `gdp_eur_per_capita`, `employees_pct_of_pop`, and five age-share columns can't be independently verified at the 18-constituency (Berlin+Hamburg) grain, so their exact ρ values inherit that uncertainty. What's left standing on its own is `foreign_population_pct` — clean, and still the single strongest predictor in the whole notebook (ρ=0.464 all constituencies, 0.431 excluding Berlin/Hamburg/Bremen, both FDR-significant) — so the deprivation/composition cluster's core claim survives using only uncontaminated data, even though its two next-strongest legs (`sgb_ii_recipients_pct`, `unemployment_total`) can't be cross-checked the same way. **Net effect of doing both 1 and 2**: nothing in §2.4's substantive conclusions depends on a contaminated column being trustworthy at face value — the political pattern was never at risk, and the deprivation pattern's weakest link is exactly the columns already flagged, not a new problem.

## 2.4e Is `pool_count` really a population-density / urban-rural proxy?

Already answered in Part 1's WHERE section, not re-derived here (that section re-ran this exact check with the same `population_density_per_km2`/`pool_count`/`racialised_rate` columns this notebook also has, so re-running it here would just reproduce the same numbers from the same source data): `pool_count` vs. `population_density_per_km2` runs ρ=-0.794 (more pools ↔ lower density -- Germany's equal-population constituency boundaries mean a sparse rural constituency must cover more land, and land, to accumulate more small pools, to reach the same population quota). Density independently out-predicts `pool_count` for `racialised_rate` (raw ρ=0.597 vs. ρ=-0.556) and survives controlling for `pool_count` (partial ρ=0.307, p<0.001), while `pool_count` controlling for density collapses to ρ=-0.169 (p=0.008) -- population density, not `pool_count`, is the more fundamental driver of the urban/rural pattern in `racialised_rate`. §2.4b's `pool_count`-based confound checks below stay as the primary control variable throughout this notebook regardless (matching Part 3's own choice, below, made on stronger multivariate grounds — AIC/BIC/cross-validation — than a single bivariate comparison like this one). (Figures refreshed to match `descriptive_analysis.ipynb`'s current WHO×WHERE capstone section — the density finding there was added/rerun after this cell was first written, and the two had drifted slightly out of sync.)

## 2.4f Re-running the AfD finding controlling for foreign population share and border status

§2.4 found `second_vote_afd_pct` negatively correlated with `racialised_rate` (ρ=-0.20), the opposite of a naive prior. §2.4 also showed this is confounded with `foreign_population_pct` (ρ=-0.54 between the two predictors themselves), and §2.4b showed the correlation collapses entirely once `pool_count` is controlled (ρ=+0.03, not significant). Neither of those checks has directly partialled out `foreign_population_pct` itself, or `is_land_border_constituency` (Part 1's corrected land-neighbour-only border flag, coastline excluded) -- border constituencies are disproportionately rural/eastern, where AfD support tends to be higher, so it's a plausible additional confound. `partial_spearman` (defined in §2.4b) only ever takes a single control variable -- it's called with `pool_count` in both its defining cell and in §2.4c's robustness check, and nowhere else -- so we add a small generalization, `partial_spearman_multi`, using the identical rank-residualization logic extended to more than one control column via a *multiple* (not simple) OLS regression on the ranks, rather than reaching for `pingouin` (not installed in this environment).

In [ ]:
def partial_spearman_multi(x, y, controls):
    """Generalizes partial_spearman to more than one control variable: rank-residualize x and y
    against a *multiple* OLS regression on the ranks of every control (instead of just one), then
    Spearman-correlate what's left. `controls` is a list of array-likes (may mix continuous and
    binary/dummy columns -- rankdata on a binary column just splits it into its two groups, so this
    is equivalent to using the raw dummy as a regressor)."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    rx, ry = rankdata(x), rankdata(y)
    control_ranks = np.column_stack([rankdata(np.asarray(c, dtype=float)) for c in controls])
    Z = sm.add_constant(control_ranks)
    rx_resid = sm.OLS(rx, Z).fit().resid
    ry_resid = sm.OLS(ry, Z).fit().resid
    return stats.pearsonr(rx_resid, ry_resid)

afd_raw_rho, afd_raw_p = stats.spearmanr(analysis_df["second_vote_afd_pct"], analysis_df["racialised_rate"])
afd_poolcount_rho, afd_poolcount_p = partial_spearman(
    analysis_df["second_vote_afd_pct"], analysis_df["racialised_rate"], analysis_df["pool_count"]
)
afd_foreignpop_rho, afd_foreignpop_p = partial_spearman(
    analysis_df["second_vote_afd_pct"], analysis_df["racialised_rate"], analysis_df["foreign_population_pct"]
)
afd_joint_rho, afd_joint_p = partial_spearman_multi(
    analysis_df["second_vote_afd_pct"], analysis_df["racialised_rate"],
    [analysis_df["foreign_population_pct"], analysis_df["is_land_border_constituency"].astype(float)],
)

afd_control_table = pd.DataFrame([
    {"specification": "raw (§2.4)", "rho": afd_raw_rho, "p": afd_raw_p},
    {"specification": "ctrl. pool_count (§2.4b)", "rho": afd_poolcount_rho, "p": afd_poolcount_p},
    {"specification": "ctrl. foreign_population_pct", "rho": afd_foreignpop_rho, "p": afd_foreignpop_p},
    {"specification": "ctrl. foreign_population_pct + is_land_border_constituency", "rho": afd_joint_rho, "p": afd_joint_p},
])
print(afd_control_table.round(4).to_string(index=False))


**Takeaway.** The AfD-`racialised_rate` relationship is **fully explained away, not partially** -- and adding `is_land_border_constituency` on top of `foreign_population_pct` changes essentially nothing. The raw ρ=-0.204 (p=3.8e-4) drops to a small, non-significant, sign-flipped ρ≈+0.03 (p≈0.58) once `pool_count` is controlled, and to ρ≈+0.06 (p≈0.27) once `foreign_population_pct` is controlled instead -- both controls independently kill it, and neither leaves anything resembling a genuine negative AfD effect. Adding `is_land_border_constituency` jointly with `foreign_population_pct` barely moves the estimate at all (ρ≈+0.06, p≈0.31) -- border status adds essentially no further explanatory power once foreign population share is already accounted for. This holds with the corrected, coastline-excluded border measure just as it held with the old one -- swapping definitions doesn't change the conclusion, because the joint control was already dominated by `foreign_population_pct`. **Bottom line for the reviewer's question**: there is no independent AfD-`racialised_rate` relationship left after controlling for either confound alone, let alone both together -- the original univariate correlation was never picking up an AfD-specific effect, it was standing in for `foreign_population_pct` (and, separately, for `pool_count`/venue density) the whole time.

## 2.4g Arab ethnicity-share vs. socioeconomic/political predictors

Part 1's WHO section found a constituency's `Arab` share of its own racialised reviews correlates with `racialised_rate` (ρ=+0.211, p=0.001, n=247 `reliable_n` constituencies) and is the only `ethnicity_group4` share that survives controlling for `population_density_per_km2` (ρ=+0.132, p=0.039) -- `Germany`'s link (ρ=-0.161 raw) disappears entirely under that control and `Türkiye`'s (ρ=+0.180 raw) weakens to non-significance. That check only ever controlled for density; it was never run against this notebook's own deprivation/political predictors. This section does that directly, reusing `racialised_df["ethnicity_group4"]` (added to Part 1's interchange file specifically for this check) and `run_correlations` (generalized above to take any `target_col`, not just `racialised_rate`).

In [ ]:
# reuse racialised_df's ethnicity_group4 to build each constituency's Arab share of its own
# racialised reviews, exactly as Part 1's WHO section did -- restricted to the same
# reliable_n=247 constituencies Part 1 used, since a share computed over a handful of racialised
# reviews isn't stable.
eth_share = pd.crosstab(
    racialised_df.loc[racialised_df["ethnicity_group4"] != "Unknown", "constituency_code"],
    racialised_df.loc[racialised_df["ethnicity_group4"] != "Unknown", "ethnicity_group4"],
    normalize="index",
)
reliable_df = analysis_df[analysis_df["reliable_n"]].copy()
eth_share = eth_share.reindex(reliable_df["constituency_code"]).fillna(0)
reliable_df = reliable_df.merge(
    eth_share[["Arab"]].rename(columns={"Arab": "arab_share"}), on="constituency_code"
)

# sanity check against Part 1's own number before trusting anything downstream
rho_check, p_check = stats.spearmanr(reliable_df["arab_share"], reliable_df["racialised_rate"])
print(f"sanity check vs. Part 1 (expect rho=+0.211, p=0.001): rho={rho_check:+.3f}, p={p_check:.3f}")

arab_corr = run_correlations(reliable_df, target_col="arab_share")
print("\nArab share vs. socioeconomic/political predictors (reliable_n=247):")
print(arab_corr.to_string(index=False))

# Part 1's WHO section already controlled arab_share vs. racialised_rate for
# population_density_per_km2 (rho=0.132, p=0.039) -- not re-derived here. What Part 1 couldn't
# test (it didn't have this notebook's predictors yet) is whether pool_count -- the confound
# this notebook leans on throughout, per §2.4e -- does the same thing:
rho_p, p_p = partial_spearman(reliable_df["arab_share"], reliable_df["racialised_rate"], reliable_df["pool_count"])
print(f"\narab_share vs. racialised_rate | controlling for pool_count: rho={rho_p:+.3f} p={p_p:.3f}")


**Takeaway.** The sanity check confirms this notebook's `ethnicity_group4` (replicated from Part 1's own categorization, added to the interchange file for this check) reproduces Part 1's number exactly (ρ=+0.211, p=0.001) -- the new correlations above build on trustworthy ground. **Arab share is not simply riding on general foreign-population share** -- `foreign_population_pct` itself only weakly and non-significantly correlates with `arab_share` (ρ=+0.125, p=0.049, doesn't survive FDR) -- **it's specifically tracking the deprivation/labor-market cluster**: `sgb_ii_recipients_pct` (ρ=+0.314, p=4.5e-7), `unemployment_total` (ρ=+0.296, p=2.2e-6), and `disposable_income_eur_per_capita` negatively (ρ=-0.236, p=1.8e-4) are all FDR-significant, alongside a political pattern that mirrors §2.4's own (`turnout_pct` -0.240, `second_vote_spd_pct` +0.227, `second_vote_left_pct` +0.208, `second_vote_cdu_csu_pct` -0.180, all FDR-significant; `second_vote_afd_pct` is **not** significant here, ρ=-0.101, p=0.115 -- consistent with §2.4f's finding that AfD's link to the *rate* was never Arab-specific). Note three of these five significant predictors (`sgb_ii_recipients_pct`, `unemployment_total`, `disposable_income_eur_per_capita`) are exactly the Berlin/Hamburg-contaminated columns flagged in §2.3b -- the same caveat from §2.4d applies here.

Controlling for `pool_count` attenuates the raw ρ=0.211 to roughly half (ρ=+0.123, p=0.053, borderline by the conventional cutoff) -- essentially the same attenuation Part 1 found controlling for `population_density_per_km2` instead (ρ=0.132, p=0.039), consistent with §2.4e's finding that `pool_count` and density are two views of the same urban/rural mechanism. **Arab share weakens under this control, roughly by half, but doesn't vanish the way `Germany` and `Türkiye` shares did in Part 1** -- unlike those two, it now also has a plausible non-density explanation on the table (the deprivation/labor-market cluster above), which Part 1 never tested since it only had density available at that point.

## 2.5 Group comparisons

Correlation treats each predictor as continuous and linear-in-rank; a quartile-split comparison is a complementary, more interpretable cut for a supervisor meeting ("bottom-quartile vs. top-quartile constituencies differ this much"), and doesn't assume the relationship is monotonic across the whole range. We pick the two predictors with a clear theoretical motivation for this specific comparison rather than testing all 18 again: **`foreign_population_pct`** (the most directly relevant compositional variable to racialised discourse about pool-goers) and **`second_vote_afd_pct`** (the political variable a reader will most expect to see tested directly, especially given the counter-intuitive negative correlation just found). We use **Kruskal-Wallis** (not one-way ANOVA) because `racialised_rate` is a small, right-skewed proportion, not a normally distributed measure, and quartile groups here only have ~75 constituencies each — exactly the small-N/non-normal combination Kruskal-Wallis is built for. Effect size is reported as **epsilon-squared** (rank-based, the Kruskal-Wallis analogue of eta-squared).

In [ ]:
def kruskal_with_epsilon_squared(df, group_col, value_col):
    groups = [g[value_col].values for _, g in df.groupby(group_col, observed=True)]
    h_stat, p = stats.kruskal(*groups)
    n = len(df)
    k = len(groups)
    epsilon_sq = (h_stat - k + 1) / (n - k)  # rank-based effect size analogue of eta-squared
    return h_stat, p, epsilon_sq

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, predictor, label in zip(
    axes, ["foreign_population_pct", "second_vote_afd_pct"], ["foreign population %", "AfD second-vote %"]
):
    analysis_df[f"{predictor}_quartile"] = pd.qcut(analysis_df[predictor], 4, labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"])
    h_stat, p, eps_sq = kruskal_with_epsilon_squared(analysis_df, f"{predictor}_quartile", "racialised_rate")
    print(f"{label}: Kruskal-Wallis H = {h_stat:.2f}, p = {p:.2e}, epsilon-squared = {eps_sq:.3f}")
    print(analysis_df.groupby(f"{predictor}_quartile", observed=True)["racialised_rate"].agg(["median", "mean", "count"]))
    print()

    sns.boxplot(data=analysis_df, x=f"{predictor}_quartile", y="racialised_rate", ax=ax, color="#8172B2")
    ax.set_title(f"Racialised rate by {label} quartile\nH={h_stat:.1f}, p={p:.1e}, ε²={eps_sq:.3f}")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


**Takeaway.** The two predictors tell different-strength stories. **Foreign population % shows a clean, monotonic, moderate-sized effect**: median racialised rate rises steadily from 0.97% (lowest quartile) to 1.93% (highest quartile) — roughly double — with epsilon-squared = 0.196, the largest effect size in this notebook. **AfD vote share is a much weaker and less clean effect** (epsilon-squared = 0.031, statistically significant only at p<.01 not the near-zero p-values seen elsewhere): the pattern isn't a smooth gradient across quartiles but mostly one contrast — the lowest-AfD quartile sits noticeably higher (median 1.61%) than quartiles 2-4, which are all similar to each other (~1.10-1.15%). This is consistent with §2.4's finding that the AfD correlation is likely picking up the foreign-population confound rather than an AfD-specific effect: foreign population % has both the cleaner pattern and the much larger effect size, so it's the stronger and more defensible of the two candidate predictors going into any future modeling work.

## 2.6 Spatial autocorrelation

Moved to `descriptive_analysis.ipynb`'s WHERE section (right after the choropleths), since testing whether `racialised_rate`'s clustering is real needs only the rate itself and constituency geometry -- both already available there, none of this notebook's socioeconomic/political data. Headline result, for reference: **Global Moran's I = 0.28** (z=7.91, p=0.001 over 999 permutations) -- real, significant positive spatial autocorrelation, with two named LISA hotspot clusters (Ruhr-area NRW, Berlin) that turn out to sit in exactly the higher-deprivation constituencies §2.4-2.4g above flag as correlated with the rate.

## 2.7 Dimension × political-lean cross-tabs: AfD-vote-share tercile

This is the one place where we deliberately don't just flag-and-skip because of thin cells -- the sub-dimensions need to count as real quantitative content for the paper, not only a caveat. The region × connotation/theme cut moved to `descriptive_analysis.ipynb`'s WHO section (right after its ethnicity-conditioned region test) since `region` is derived from `state` alone and needs no political data. What's left here needs this notebook's data specifically: **constituency AfD-vote-share tercile × connotation/theme** -- chosen because `second_vote_afd_pct` is the political variable most likely to be asked about directly, despite §2.4-2.4f already showing its univariate correlation with the *rate* is confounded by foreign population share/venue density. This does **not** carry that confound over automatically -- this cross-tab asks a different question (given a review is already racialised, does its *content* differ by political context?), so it's still worth running rather than assuming the answer. We report cell counts alongside every test and use Fisher's exact test (not just chi-square) wherever expected cell counts drop below 5, rather than excluding those results.

In [ ]:
def chi2_cramers_v(ct):
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    min_expected = expected.min()
    return chi2, p, dof, v, min_expected


In [ ]:
# --- constituency AfD-vote-share tercile x connotation / theme (secondary, exploratory cut) ---
analysis_df["afd_tercile"] = pd.qcut(
    analysis_df["second_vote_afd_pct"], 3, labels=["Low AfD", "Mid AfD", "High AfD"]
)
afd_tercile_lookup = analysis_df.set_index("constituency_code")["afd_tercile"]
racialised_df = racialised_df.assign(afd_tercile=racialised_df["constituency_code"].map(afd_tercile_lookup))

print("racialised reviews per AfD tercile (constituency-level tercile, broadcast to reviews):")
print(racialised_df["afd_tercile"].value_counts())
print()

ct_afd_conn = pd.crosstab(racialised_df["afd_tercile"], racialised_df["connotation"].apply(lambda x: x[0]))
chi2, p, dof, v, min_exp = chi2_cramers_v(ct_afd_conn)
print("AfD tercile x connotation:")
print(ct_afd_conn)
print(f"chi2={chi2:.1f}, dof={dof}, p={p:.2e}, Cramer's V={v:.3f}, min expected count={min_exp:.1f}\n")

afd_theme = racialised_df[["review_id", "afd_tercile", "theme"]].explode("theme")
ct_afd_theme = pd.crosstab(afd_theme["afd_tercile"].to_numpy(), afd_theme["theme"].to_numpy())
chi2, p, dof, v, min_exp = chi2_cramers_v(ct_afd_theme)
print("AfD tercile x theme:")
print(ct_afd_theme)
print(f"chi2={chi2:.1f}, dof={dof}, p={p:.2e}, Cramer's V={v:.3f}, min expected count={min_exp:.1f}")
n_thin_cells = (ct_afd_theme.values < 5).sum()
print(f"cells with raw count < 5: {n_thin_cells} / {ct_afd_theme.size}")


**Takeaway.** Across both cuts, **connotation doesn't shift with political/regional context, but theme does** — a genuinely informative and fairly clean pattern:

- **Region × connotation**: not significant (χ²=6.7, p=0.15, V=0.021). **Region × theme**: significant but small (χ²=84.7, p=6.2e-10, V=0.056) — one cell (`Other` theme in Berlin) has an expected count of 4.5, just under the conventional 5 threshold, so that specific cell should be read cautiously, but the overall test isn't driven by it alone.
- **AfD tercile × connotation**: not significant (χ²=4.2, p=0.38, V=0.016) — all cell counts are comfortably large (min expected 101.3), so this is a clean null, not an underpowered one. **AfD tercile × theme**: significant, small effect (χ²=70.4, p=1.6e-7, V=0.051), and here every cell clears the count-5 threshold (min expected 37, no thin cells at all) — the cleanest-powered test of the four. The clearest single contrast: `Security` as a theme is notably more common in Low-AfD-tercile reviews (102) than Mid (73) or High-AfD-tercile (47) — roughly 2x more frequent at the low end than the high end.

**Overall reading**: once a review is already racialised, *how negative/neutral/positive* it reads doesn't depend on the region or the local political lean — but *what it's about* (crowd, staff, security, etc.) does, modestly, in both cuts. This is a legitimately new, adequately-powered quantitative finding about the sub-dimensions (not just a caveated non-result), and it's a useful qualifier on Part 2's earlier area-level correlations: those were about *how often* racialised discourse occurs; this section shows its *content* is comparatively stable by connotation but does vary somewhat by theme across political/regional context — worth a closer qualitative read on the `Security`/`Safety` themes specifically, which also stood out in §1.6's full-sample theme×connotation analysis.

## 2.8 Closing synthesis: candidate refined RQs for the supervisor meeting

**What's well-supported right now, across both notebooks:**
- Overall racialised-review prevalence (1.54% of social reviews, tight CI) and its constituency/state distribution, backed by 83% constituency coverage (247/299, at Part 1's current `racialised_reviews >= 10` reliability bar) at a defensible reliability bar based on racialised/social-review counts, not pool count (`descriptive_analysis.ipynb`'s WHERE section).
- A real, complete-data (99.94% timestamp coverage after recovery) temporal rise from 0.92% (2021) to 2.11% (2025).
- Genuine, statistically confirmed spatial clustering (Global Moran's I = 0.28, p=0.001) in two named hotspot regions: Ruhr-area NRW and Berlin (computed in `descriptive_analysis.ipynb`'s WHERE section, right after the choropleths -- moved there from an earlier draft of this notebook since it needs none of Part 2's data).
- A deprivation cluster (`sgb_ii_recipients_pct`, `foreign_population_pct`, `unemployment_total`) that survives controlling for the strongest confound found (`pool_count`/venue density, §2.4b) — attenuated to about half its univariate magnitude (ρ≈0.23-0.28 partial vs. 0.43-0.47 univariate) but still FDR-significant. `turnout_pct` also survives essentially unchanged.
- A specific, now doubly-confirmed caution around the political predictors: `second_vote_afd_pct`'s univariate correlation (ρ=−0.20) collapses to ρ=+0.03 (p=0.58, not significant) once `pool_count` is controlled — an even more decisive confound than the §2.4 check against `foreign_population_pct` alone found. **This was never an independent AfD effect.**
- A broader methodological finding from §2.4b that reshapes how to read *all* of §2.4: almost every predictor (age structure, `second_vote_left_pct`/`greens_pct`/`cdu_csu_pct`, GDP, employment share) is itself strongly correlated with `pool_count` (|ρ| 0.32-0.57), and loses FDR significance once that's controlled for — meaning most of §2.4's univariate correlations, taken individually, overstate how much each variable independently matters. Only the deprivation cluster and turnout hold up to this specific check.
- A clean split in how the racialised reviews' *sub-dimensions* relate to context: `connotation` is stable across region/political-lean, `theme` shifts modestly (small but adequately powered effects, notably around `Security`).
- **A new, specific compositional lead** (§2.4g, following up on Part 1's WHO section): a constituency's `Arab` share of its own racialised reviews tracks the deprivation/labor-market cluster directly (`sgb_ii_recipients_pct` ρ=0.31, `unemployment_total` ρ=0.30, `disposable_income_eur_per_capita` ρ=-0.24, all FDR-significant) and a political pattern mirroring §2.4's own -- but **not** `foreign_population_pct` (ρ=0.13, not FDR-significant) or `second_vote_afd_pct` (ρ=-0.10, ns). Attenuates to roughly half under `pool_count`/density control (ρ≈0.12, borderline p≈0.05-0.06) rather than vanishing, unlike `Germany`/`Türkiye` share in Part 1 -- a genuine candidate covariate for Part 3's multivariate model below, not just a density proxy.
- **A data-provenance fix, not a finding**: `const_agg` was rebuilt since this notebook was first written, replacing the coastline-inclusive `is_border_constituency` with the corrected, land-neighbour-only `is_land_border_constituency`. Re-run under the corrected flag, §2.4f's conclusion is unchanged (border status still adds nothing once `foreign_population_pct` is controlled) -- flagged here so the swap doesn't get mistaken for a substantive result.
- **§2.4d's robustness check was extended to also report a "clean predictors only" view** (§2.3b's 10 Berlin/Hamburg-contaminated columns dropped entirely, not just Berlin/Hamburg/Bremen constituencies dropped): the political-lean pattern is fully clean by construction and unaffected either way; the deprivation story's one clean leg, `foreign_population_pct`, remains the strongest predictor in the notebook (ρ=0.46) on its own, but `sgb_ii_recipients_pct`/`unemployment_total`/income can't be independently cross-checked at the Berlin+Hamburg grain.

**Three sharper RQs this supports, in order of how supported they already are:**

1. **"Where and when does racialised discourse concentrate, and is that concentration spatially structured or incidental?"** — fully supported now with Moran's I/LISA + the temporal trend; could be written up close to as-is.
2. **"What structural (socioeconomic, demographic) features of a place are associated with racialised-discourse prevalence, net of confounding between those features — including venue density itself?"** — §2.4b already shows this can't be answered with one-at-a-time partial correlations (too many predictors are mutually confounded with `pool_count` and likely each other); it needs a multivariate model (negative-binomial GLM with a review-volume offset, `pool_count`/venue-density included as a covariate rather than partialled out separately) -- that's Part 3, directly below.
3. **"Does the *content* of racialised discourse (not just its frequency) vary systematically by place?"** — §2.7 gives an initial, adequately-powered answer (theme shifts modestly, connotation doesn't), but this was only tested against two political/regional cuts; a fuller treatment (e.g. against the deprivation variables that were strongest in RQ2, or against `approach`/`context` which weren't tested here) is a natural extension once the sub-dimension labels have been through the planned inter-annotator-agreement check — worth prioritising IAA on `theme` specifically, since it's the dimension carrying the geographic signal.

**What Part 2 above deliberately does not do**, so Part 3's contribution below is unambiguous: no multivariate/joint model (that's Part 3's job), no causal language anywhere (every Part 2 result is explicitly framed as area-level/ecological), and no dimension-level constituency-grain modeling (only region- and tercile-level, which have adequate power; true constituency-grain dimension analysis would need either a much larger racialised-review sample or a deliberate oversampling strategy).

## Part 3 — Predictive modelling: racialised rate as a function of socioeconomic/political structure

Continues directly in this notebook: reuses `analysis_df`, `PREDICTOR_COLS`, `political`, and `const_agg` as already built in Part 2 above (no reload, no re-derivation) -- a single multivariate model with `pool_count` included as a covariate, rather than the one-at-a-time partial correlations §2.4b had to rely on. This directly follows §2.8's RQ2 spec: *"a negative-binomial GLM with a review-volume offset, `pool_count`/venue-density included as a covariate rather than partialled out separately."* This is a preliminary pass on the working assumption that the IAA and robustness work already completed is sufficient to trust the `racialised` label going into a model -- both the label and the model itself are expected to be iterated on further.


## 3.1 Model specification

Two choices, each following directly from what Parts 1-2 already established:

- **Outcome is the count, not the rate, with an offset.** `racialised_reviews` (not `racialised_rate`) is the response, with `log(social_reviews)` as an offset — the standard way to model a rate that comes from a variable-sized denominator (analogous to person-time in epidemiology). It lets the model use the count's actual sampling variance instead of treating `racialised_rate` as a continuous measurement with constant precision, which it isn't: a constituency with 300 social reviews and one with 3,000 have very different sampling noise around the same rate.
- **Negative binomial, not Poisson.** §2.6 already found genuine spatial clustering (Moran's I = 0.28, p = 0.001) — clustering implies extra between-constituency variance beyond what a Poisson model's mean-equals-variance assumption allows. We fit the Poisson model first specifically to demonstrate this overdispersion numerically before moving to NB2, rather than asserting it.

What still needs deciding is *which* covariates go in alongside `pool_count` — §3.2 below.

In [ ]:
X_full = sm.add_constant(analysis_df[["pool_count"] + PREDICTOR_COLS])
y = analysis_df["racialised_reviews"]
offset = analysis_df["log_social_reviews"]

poisson_full = sm.GLM(y, X_full, family=sm.families.Poisson(), offset=offset).fit()
dispersion_full = poisson_full.pearson_chi2 / poisson_full.df_resid
print(f"Poisson GLM (pool_count + all {len(PREDICTOR_COLS)} predictors): "
      f"Pearson chi2 / df = {dispersion_full:.2f} (>>1 => overdispersed, NB2 needed either way)")

Pearson chi2/df = 7.99, well above the 1.0 a well-specified Poisson model would show — confirms overdispersion, consistent with §2.6's spatial clustering finding, and is the empirical justification for using NB2 rather than Poisson throughout the rest of this notebook (the reduced-set Poisson model in §3.3 shows the same pattern, chi2/df = 9.46).

## 3.2 Predictor set: why not just put all 18 in?

§2.4b/§2.8 called for "`pool_count` as a covariate, not partialled out separately" precisely *because* the §2.4 predictors are themselves heavily intercorrelated with `pool_count` and with each other (deprivation, age structure, and vote share all move together). The natural first attempt is therefore the maximal one: `pool_count` + all 18 predictors in a single NB2. We check that model's viability with VIF (variance inflation factor) *before* trusting any coefficient from it — a VIF above ~10 means a predictor's estimated effect is essentially undetermined by the data, drowned out by its correlation with the other covariates in the model, not evidence of a null effect.

In [ ]:
vif_full = pd.DataFrame({
    "predictor": X_full.columns,
    "vif": [variance_inflation_factor(X_full.values, i) for i in range(X_full.shape[1])],
}).drop(index=0)  # drop the constant term's VIF, which is uninformative here
vif_full.sort_values("vif", ascending=False)

Confirms the concern directly: VIF exceeds 10 for **11 of the 19 covariates**, topping out at **79.7 for `second_vote_afd_pct`**, 58.6 for `second_vote_cdu_csu_pct`, and 38.4 for `second_vote_greens_pct` — the vote-share variables are so intercorrelated with each other (they're shares of the same total, on top of §2.4b's already-established correlation with `pool_count`) that no single one of them has an independently estimable effect in a model that includes all of them at once. `pool_count` itself has a comfortable VIF of 2.3 — it's the combination of many mutually collinear vote-share/deprivation variables that's unusable, not `pool_count`'s inclusion.

Given that, we don't fit the 19-covariate NB2 as the primary model at all — attempting it directly confirms the concern: the MLE fails to converge (Hessian inversion fails, so no standard errors are even computable) and 5-fold cross-validation on it produces predictions that are numerically unstable across folds. That's not a coding bug, it's the expected consequence of asking an MLE to separate the effects of 19 covariates, several pairwise/multiply collinear, from n=299 constituencies.

Instead we use the **6-predictor set §2.4b already validated as robust to the `pool_count` confound** — `sgb_ii_recipients_pct`, `foreign_population_pct`, `unemployment_total`, `turnout_pct`, `second_vote_spd_pct`, `second_vote_bsw_pct` (the only ones that stayed FDR-significant once `pool_count` was partialled out one at a time). Moving from "one-at-a-time partial correlation" to "one joint multivariate model" only for *this* set is the actual intuitive next step from §2.4b/§2.8 — it keeps the covariate-to-observation ratio sane (7 covariates, n=299) while still jointly modelling the interactions those 6 predictors have with each other and with `pool_count`, which one-at-a-time partial correlation could not do.

In [ ]:
REDUCED_PREDICTOR_COLS = [
    "sgb_ii_recipients_pct", "foreign_population_pct", "unemployment_total",
    "turnout_pct", "second_vote_spd_pct", "second_vote_bsw_pct",
]
MODEL_COLS = ["pool_count"] + REDUCED_PREDICTOR_COLS

X = sm.add_constant(analysis_df[MODEL_COLS])

vif_reduced = pd.DataFrame({
    "predictor": X.columns,
    "vif": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
}).drop(index=0)
vif_reduced.sort_values("vif", ascending=False)

Much better, but not perfect: `sgb_ii_recipients_pct` (VIF 26.8) and `unemployment_total` (VIF 25.8) are still highly collinear with each other — unsurprising, since both are deprivation measures — while `pool_count` and the remaining four predictors are all comfortably below 4. This is a known limitation of this pass (flagged again in §3.7): the model below can jointly estimate "the deprivation cluster's" effect reasonably well, but can't cleanly separate `sgb_ii_recipients_pct`'s individual contribution from `unemployment_total`'s.

## 3.3 Fitting the negative binomial model on the reduced set

In [ ]:
poisson_model = sm.GLM(y, X, family=sm.families.Poisson(), offset=offset).fit()
dispersion = poisson_model.pearson_chi2 / poisson_model.df_resid
print(f"Poisson GLM (reduced set): Pearson chi2 / df = {dispersion:.2f}")

nb_full = NegativeBinomial(y, X, offset=offset).fit(disp=False)
print(nb_full.summary())

In [ ]:
irr_table = pd.DataFrame({
    "coef": nb_full.params,
    "irr": np.exp(nb_full.params),
    "irr_ci_lo": np.exp(nb_full.conf_int()[0]),
    "irr_ci_hi": np.exp(nb_full.conf_int()[1]),
    "p": nb_full.pvalues,
}).drop(index=["alpha", "const"], errors="ignore")
irr_table.sort_values("p")

**Takeaway.** The model converges cleanly this time (`converged: True`), and is overall highly significant (LLR p = 6.8e-30 vs. the null offset-only model), with a modest pseudo-R² of 0.066 — expected for a preliminary ecological model of a rare, noisy outcome. Two covariates are clearly significant once everything is modelled jointly:

- **`foreign_population_pct`** (IRR = 1.034, 95% CI [1.019, 1.049], p < 0.0001): each 1 percentage-point increase in foreign-population share is associated with a 3.4% higher expected racialised-review rate, holding `pool_count` and the other five predictors constant. This is the one predictor that has now survived three separate checks — §2.4's univariate correlation (ρ=0.46), §2.4b's `pool_count`-partialled correlation (ρ=0.28), and this joint multivariate model — making it the most defensible finding across all three notebooks so far.
- **`pool_count`** (IRR = 0.989, 95% CI [0.980, 0.997], p = 0.010): each additional pool in a constituency is associated with a ~1.1% lower expected rate, confirming §2.4b's confound finding survives being modelled as a covariate rather than partialled out.

Everything else is weaker than §2.4b's partial correlations suggested: `sgb_ii_recipients_pct` (p=0.070) and `turnout_pct` (p=0.096) are only marginal, and **`unemployment_total`'s sign flips to negative** (coef=-0.068, p=0.317, not significant) — almost certainly an artifact of its VIF=25.8 collinearity with `sgb_ii_recipients_pct` (§3.2) rather than a real reversal of the §2.4/§2.4b positive association; the two deprivation variables are fighting over the same shared variance once modelled together. `second_vote_spd_pct` and `second_vote_bsw_pct` are both null here (p=0.849, p=0.461) — §2.4b's partial correlations for these two (ρ=0.14, ρ=0.15) don't survive being placed in a joint model alongside the stronger deprivation/composition predictors.

In [ ]:
X_reduced_null = sm.add_constant(analysis_df[["pool_count"]])
nb_pool_only = NegativeBinomial(y, X_reduced_null, offset=offset).fit(disp=False)

lr_stat = 2 * (nb_full.llf - nb_pool_only.llf)
df_diff = len(nb_full.params) - len(nb_pool_only.params)
p_lr = stats.chi2.sf(lr_stat, df_diff)

print(f"pool_count-only NB2: log-likelihood = {nb_pool_only.llf:.1f}")
print(f"pool_count + {len(REDUCED_PREDICTOR_COLS)} predictors NB2: log-likelihood = {nb_full.llf:.1f}")
print(f"likelihood-ratio test: LR = {lr_stat:.1f}, df = {df_diff}, p = {p_lr:.2e}")

**Takeaway.** The 6 predictors jointly add real explanatory power beyond `pool_count` alone: LR = 77.2 on 6 df, p = 1.3e-14. This directly answers §2.8's RQ2 — "is there a real multivariate model here, or does pool_count/venue density explain the pattern by itself" — and confirms that no, deprivation and composition variables (mainly `foreign_population_pct`) explain something venue density alone does not.

## 3.3b Does the same model explain the racial/all-reviews rate?

§2.4's rate-comparison check found the two racialised-rate definitions (`/social` vs. `/all text reviews`) tell an almost identical correlational story. This repeats that check at the regression stage: same count (`racialised_reviews`), same `MODEL_COLS` (VIF is a property of the predictor matrix alone, §3.2, so the reduced set doesn't need re-deriving), only the offset changes -- `log_total_text_reviews` in place of `log_social_reviews` -- to see whether the joint model tells a different story once the denominator is diluted with reviews that were never in scope for racialised-discourse coding.

In [ ]:
offset_rate2 = analysis_df["log_total_text_reviews"]

nb_full_rate2 = NegativeBinomial(y, X, offset=offset_rate2).fit(disp=False)
print(nb_full_rate2.summary())

In [ ]:
irr_table_rate2 = pd.DataFrame({
    "coef": nb_full_rate2.params,
    "irr": np.exp(nb_full_rate2.params),
    "irr_ci_lo": np.exp(nb_full_rate2.conf_int()[0]),
    "irr_ci_hi": np.exp(nb_full_rate2.conf_int()[1]),
    "p": nb_full_rate2.pvalues,
}).drop(index=["alpha", "const"], errors="ignore")

model_comparison = irr_table[["irr", "p"]].join(
    irr_table_rate2[["irr", "p"]], lsuffix="_social", rsuffix="_all"
)
model_comparison.sort_values("p_social")

In [ ]:
nb_pool_only_rate2 = NegativeBinomial(y, X_reduced_null, offset=offset_rate2).fit(disp=False)

lr_stat_rate2 = 2 * (nb_full_rate2.llf - nb_pool_only_rate2.llf)
p_lr_rate2 = stats.chi2.sf(lr_stat_rate2, len(nb_full_rate2.params) - len(nb_pool_only_rate2.params))
print(f"pool_count + {len(REDUCED_PREDICTOR_COLS)} predictors NB2 (racial/all offset): "
      f"log-likelihood = {nb_full_rate2.llf:.1f}")
print(f"likelihood-ratio test vs. pool_count-only: LR = {lr_stat_rate2:.1f}, p = {p_lr_rate2:.2e}")

**Takeaway.** Swapping the offset from `log_social_reviews` to `log_total_text_reviews` barely changes the model: the same two covariates are significant, with the same signs and similar magnitudes -- `foreign_population_pct` (IRR 1.032 vs. 1.034, p<0.0001 both) and `pool_count` (IRR 0.987 vs. 0.989, p=0.004 vs. p=0.010) -- and every other covariate stays non-significant in both fits (`sgb_ii_recipients_pct` p=0.099 vs. 0.070, `turnout_pct` p=0.131 vs. 0.096, `unemployment_total` p=0.370 vs. 0.317, `second_vote_spd_pct`/`second_vote_bsw_pct` both null). The joint LR test against the `pool_count`-only null is highly significant either way (LR=72.2, p=1.4e-13 for `/all`, vs. LR=77.2, p=1.3e-14 for `/social`).

This confirms at the regression stage what §2.4's correlation-level check already found: the two racialised-rate definitions are redundant here, not just correlated. Nothing downstream (§3.4 onward) needs to be re-run for `racialised_rate_of_all` -- Part 1's `/social` choice is the one worth carrying forward, and the rest of this notebook continues to use it as the primary DV.

## 3.4 Does the model actually predict better, out-of-sample?

§3.3 tests whether the predictors are jointly *significant* in-sample; that's a different question from whether they improve *predictive* performance on constituencies the model hasn't seen. We run 5-fold cross-validation comparing the `pool_count`-only model against the reduced model, scoring each fold's held-out predictions on the rate scale (predicted rate vs. actual `racialised_rate`) with mean absolute error — a simple, interpretable predictive check appropriate for a preliminary pass on n=299.

In [ ]:
def cv_mae(cols, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    X_full_cv = sm.add_constant(analysis_df[cols])
    y_full = analysis_df["racialised_reviews"]
    offset_full = analysis_df["log_social_reviews"]
    rate_actual = analysis_df["racialised_rate"].values
    errors = []
    for train_idx, test_idx in kf.split(np.arange(len(analysis_df))):
        model = NegativeBinomial(
            y_full.iloc[train_idx], X_full_cv.iloc[train_idx], offset=offset_full.iloc[train_idx]
        ).fit(disp=False)
        pred_rate = np.exp(X_full_cv.iloc[test_idx].values @ model.params.drop("alpha").values)
        errors.append(np.abs(pred_rate - rate_actual[test_idx]))
    return np.concatenate(errors)

mae_pool_only = cv_mae(["pool_count"])
mae_reduced = cv_mae(MODEL_COLS)

print("5-fold CV mean absolute error (percentage points of racialised_rate):")
print(f"  pool_count-only model: {mae_pool_only.mean() * 100:.3f} pp")
print(f"  reduced model:         {mae_reduced.mean() * 100:.3f} pp")

**Takeaway.** The reduced model beats the `pool_count`-only model out-of-sample too, not just in-sample: 0.610pp mean absolute error vs. 0.691pp — an ~12% relative reduction. The improvement is real but modest, consistent with the in-sample picture above: one strong predictor (`foreign_population_pct`) plus a weak confound-adjustment covariate (`pool_count`), rather than a large multi-predictor gain. That's expected for a first predictive pass on a rare, noisy ecological outcome with n=299, and it sets a concrete baseline (0.61pp MAE) that future iterations of this model can be compared against.

## 3.4b Does `population_density_per_km2` outperform `pool_count` as the density covariate?

`02_socioeconomic_political_relations.ipynb` §2.7-2.8 added `population_density_per_km2` to the data pipeline and found it out-predicts `pool_count` in raw correlation terms (density vs `racialised_rate` ρ=0.54 vs `pool_count` vs `racialised_rate` ρ=-0.49), while the two are themselves strongly collinear (Spearman ρ=-0.80: more pools ↔ lower density, confirming `pool_count` is essentially a noisy proxy for density). That reframes a question this notebook's §3.1-3.3 model didn't ask: is `pool_count` in `MODEL_COLS` actually the best choice of density covariate, or would swapping in `population_density_per_km2` (or including both) fit better?

We compare three specifications, all using the same 6-predictor `REDUCED_PREDICTOR_COLS` set from §3.2:
1. **`pool_count`-based** (the existing §3.3 model, `nb_full`).
2. **Density-only**: `population_density_per_km2` in place of `pool_count`.
3. **Both together**: `pool_count` and `population_density_per_km2` jointly (checking whether their ρ=-0.80 collinearity makes this poorly identified).

First, though: `population_density_per_km2` ranges from ~37/km² in rural constituencies to ~12,190/km² in dense Berlin seats, which is likely far more right-skewed than `pool_count` and may need a log-transform before it can be used as a sane linear covariate in the GLM's linear predictor.

In [ ]:
print("Skewness (right-skew if >> 0):")
print(f"  pool_count:                          {analysis_df['pool_count'].skew():.3f}")
print(f"  population_density_per_km2 (raw):    {analysis_df['population_density_per_km2'].skew():.3f}")
print(f"  log(population_density_per_km2):     {np.log(analysis_df['population_density_per_km2']).skew():.3f}")

analysis_df["log_population_density"] = np.log(analysis_df["population_density_per_km2"])

print(f"\nSpearman rho, pool_count vs population_density_per_km2 (raw): "
      f"{stats.spearmanr(analysis_df['pool_count'], analysis_df['population_density_per_km2'])[0]:.3f}")

`population_density_per_km2`'s skewness (~3.3) is roughly 3.7x `pool_count`'s (~0.9) and drops to a much more GLM-friendly ~0.6 once log-transformed — confirming the concern flagged in the task spec. `pool_count` itself is mildly right-skewed but nowhere near as extreme, and it's already working fine in the existing §3.3 model (IRR, CIs, and diagnostics all sensible), so per the "don't change what isn't broken" principle it is left untransformed. `population_density_per_km2` is used in **log form** (`log_population_density`) throughout this section — confirmed necessary below, not just a precaution: the raw-scale density model fails to converge.

In [ ]:
REDUCED_PLUS = REDUCED_PREDICTOR_COLS  # sgb_ii_recipients_pct, foreign_population_pct, unemployment_total,
                                        # turnout_pct, second_vote_spd_pct, second_vote_bsw_pct

def fit_reduced(cols):
    Xv = sm.add_constant(analysis_df[cols])
    return Xv, NegativeBinomial(y, Xv, offset=offset).fit(disp=False)

# Variant 1: pool_count-based -- this is exactly nb_full from §3.3, refit here under a
# variant-specific name for a clean side-by-side comparison.
X_v_pool, nb_v_pool = fit_reduced(["pool_count"] + REDUCED_PLUS)

# Variant 1b: does the raw (untransformed) density scale even converge? (diagnostic only)
X_v_density_raw, nb_v_density_raw = fit_reduced(["population_density_per_km2"] + REDUCED_PLUS)
print(f"Raw-scale density model converged: {nb_v_density_raw.mle_retvals.get('converged')}")

# Variant 2: density-only (log scale)
X_v_density, nb_v_density = fit_reduced(["log_population_density"] + REDUCED_PLUS)

# Variant 3: both together (log density + pool_count)
X_v_both, nb_v_both = fit_reduced(["pool_count", "log_population_density"] + REDUCED_PLUS)

vif_both = pd.DataFrame({
    "predictor": X_v_both.columns,
    "vif": [variance_inflation_factor(X_v_both.values, i) for i in range(X_v_both.shape[1])],
}).drop(index=0)
vif_both.sort_values("vif", ascending=False)

**Convergence and VIF.** The raw-scale density model fails to converge (`converged: False`) -- confirming the skew concern isn't just cosmetic, it breaks the MLE outright, exactly as the log-transform diagnostic above predicted. The log-scale density model converges cleanly.

For the "both together" spec, `log_population_density`'s VIF is **7.24** against `pool_count`'s 2.44 -- elevated, and by far the third-highest VIF in that model after the already-known `sgb_ii_recipients_pct`/`unemployment_total` pair (~26-29 each, unchanged from §3.2), but it stays under the conventional VIF>10 danger threshold. So the two aren't so collinear that a joint model is uninterpretable the way the 19-covariate model in §3.2 was -- but as the LR test below shows, "not fatally collinear" isn't the same as worth including both.

In [ ]:
def cv_mae_variant(cols, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    Xcv = sm.add_constant(analysis_df[cols])
    rate_actual = analysis_df["racialised_rate"].values
    errors = []
    for train_idx, test_idx in kf.split(np.arange(len(analysis_df))):
        model = NegativeBinomial(y.iloc[train_idx], Xcv.iloc[train_idx], offset=offset.iloc[train_idx]).fit(disp=False)
        pred_rate = np.exp(Xcv.iloc[test_idx].values @ model.params.drop("alpha").values)
        errors.append(np.abs(pred_rate - rate_actual[test_idx]))
    return np.concatenate(errors)

variant_cols = {
    "pool_count-based":  ["pool_count"] + REDUCED_PLUS,
    "density-only (log)": ["log_population_density"] + REDUCED_PLUS,
    "both together":      ["pool_count", "log_population_density"] + REDUCED_PLUS,
}
variant_models = {"pool_count-based": nb_v_pool, "density-only (log)": nb_v_density, "both together": nb_v_both}

# LR tests: "both" nests both single-covariate variants (drop one term each), so both are
# directly testable; pool_count-only vs density-only are non-nested (can't LR test directly).
lr_add_density = 2 * (nb_v_both.llf - nb_v_pool.llf)
p_add_density = stats.chi2.sf(lr_add_density, 1)
lr_add_pool = 2 * (nb_v_both.llf - nb_v_density.llf)
p_add_pool = stats.chi2.sf(lr_add_pool, 1)

rows = []
for name, cols in variant_cols.items():
    m = variant_models[name]
    mae = cv_mae_variant(cols).mean() * 100
    fp_row = {
        "spec": name, "AIC": m.aic, "BIC": m.bic, "llf": m.llf,
        "cv_mae_pp": mae,
        "foreign_population_pct_IRR": np.exp(m.params["foreign_population_pct"]),
        "foreign_population_pct_p": m.pvalues["foreign_population_pct"],
    }
    rows.append(fp_row)
comparison = pd.DataFrame(rows).set_index("spec")
print(comparison.round(4))

print(f"\nLR test, adding log_population_density to pool_count-based model: "
      f"LR={lr_add_density:.3f}, df=1, p={p_add_density:.4f}")
print(f"LR test, adding pool_count to density-only model: "
      f"LR={lr_add_pool:.3f}, df=1, p={p_add_pool:.4f}")

**Takeaway -- three-way comparison and recommendation.**

| Spec | AIC | BIC | 5-fold CV MAE (pp) | `foreign_population_pct` IRR (p) |
|---|---|---|---|---|
| `pool_count`-based (current, §3.3) | 2200.6 | 2233.9 | 0.610 | 1.034 (p<0.0001) |
| Density-only (log) | 2202.8 | 2236.1 | 0.613 | 1.026 (p=0.014) |
| Both together | 2201.6 | 2238.6 | 0.615 | 1.027 (p=0.010) |

Three findings:

1. **`foreign_population_pct` is robust across every specification tried.** IRR ranges narrowly from 1.026 to 1.034 (a 2.6-3.4pp effect per 1pp of foreign-population share), and it is significant at p<0.05 in all three, p<0.0001 in the primary spec. Swapping the density covariate does not touch this finding -- exactly what notebook 02's independent confirmation would predict.
2. **The `pool_count`-based model still edges out both alternatives on every metric** (lowest AIC/BIC, lowest CV MAE), though the margins are small (AIC differs by ~1-2 points, CV MAE by ~0.005pp) -- these are not decisive wins, more like a photo finish. Consistent with that, the LR tests confirm neither addition earns its extra parameter: adding `log_population_density` to the `pool_count`-based model is not significant (LR=0.98, p=0.32), and adding `pool_count` to the density-only model is only marginal (LR=3.20, p=0.074).
3. **We do not adopt the "both together" specification.** Even though its VIF (7.24 for `log_population_density`) stays just under the conventional danger threshold and the model is technically estimable, the LR tests above show it isn't buying real explanatory power over either single-covariate variant -- it's just spending a degree of freedom on two measures of essentially the same underlying construct (pool density vs population density, ρ=-0.80).

**Recommendation: keep `pool_count` as the primary density covariate** (`MODEL_COLS` and `nb_full` in §3.2-3.3 are unchanged). It has a marginal but consistent edge in fit and out-of-sample prediction, and -- unlike `population_density_per_km2` -- it doesn't need a log-transform to be usable, which keeps the existing IRR-based interpretation simple. This is a closer call than notebook 02's raw correlation numbers alone would suggest (density "wins" ρ=0.54 vs -0.49 there), and it's worth being honest about why: once `foreign_population_pct` and the other five predictors are already in the model, whichever density measure is added contributes comparatively little either way, and `pool_count`'s advantage is small enough that a future iteration with a cleaner or updated density measure could plausibly flip this. For now, `population_density_per_km2` (log-transformed) is flagged as a close, theoretically well-motivated second choice worth revisiting rather than a decisively rejected alternative -- and the log-transform is a **permanent requirement if it's ever adopted**, not just a one-off diagnostic check for this comparison (the raw scale genuinely fails to converge).

## 3.5 Robustness check: restricting to the `reliable_n` constituencies

Same check §2.4 ran for the univariate correlations -- refit the model restricted to constituencies clearing §1.2's reliability bar. **Note:** `descriptive_analysis.ipynb` tightened this bar from `social_reviews>=100 AND racialised_reviews>=5` to `social_reviews>=100 AND racialised_reviews>=10` -- so the reliable subset is now smaller than it was when this notebook was first built (247/299 pass, not the 288/299 the bar used to yield; that number is computed directly from `analysis_df["reliable_n"]` just below rather than hardcoded, so it always reflects whatever the upstream notebook currently produces). We compare coefficients to the all-299 model to see whether any result is being driven by the (now 52, not 11) thinner constituencies excluded by the stricter bar.

In [ ]:
n_reliable = int(analysis_df["reliable_n"].sum())
print(f"reliable_n constituencies: {n_reliable} / {len(analysis_df)}")

reliable_df = analysis_df[analysis_df["reliable_n"]]
X_rel = sm.add_constant(reliable_df[MODEL_COLS])
y_rel = reliable_df["racialised_reviews"]
offset_rel = reliable_df["log_social_reviews"]

nb_reliable = NegativeBinomial(y_rel, X_rel, offset=offset_rel).fit(disp=False)

coef_compare = pd.DataFrame({
    "coef_all_299": nb_full.params,
    f"coef_reliable_{n_reliable}": nb_reliable.params,
}).drop(index="alpha", errors="ignore")
coef_compare["shift"] = coef_compare[f"coef_reliable_{n_reliable}"] - coef_compare["coef_all_299"]
coef_compare.sort_values("shift", key=np.abs, ascending=False)

**Takeaway.** All 7 coefficients are stable between the 299-constituency and the (now 247, corrected from a stale 288 -- see §3.5 header) `reliable_n` fits -- the largest shifts are `unemployment_total` (-0.039) and `sgb_ii_recipients_pct` (+0.026), both larger in absolute terms than the old 288-subset check reported (expected, since the stricter `>=10` bar now excludes 52 constituencies rather than 11, a much bigger and more different subset), but still small relative to their standard errors in §3.3 and every coefficient keeps its sign. None of §3.3's findings are being driven by the thin constituencies the reliability bar screens out, matching the pattern §2.4's correlation robustness check already found -- the stricter threshold changes *which* constituencies are excluded but not the substantive conclusion.

## 3.6 Do model residuals still show spatial clustering?

§2.6 found genuine spatial clustering in the raw `racialised_rate` (Moran's I = 0.28). If the covariates in this model fully explained that clustering, the model's residuals should look spatially random; if Moran's I on the residuals is still significant, that's a sign the NB2 specification is missing something structurally spatial (e.g. a spatial lag or a state-level random effect) — a concrete, flagged item for the next iteration, not something to paper over by adding more predictors to this pass.

In [ ]:
from libpysal.weights import Queen
from esda.moran import Moran

mu_full = np.exp(X.values @ nb_full.params.drop("alpha").values + offset.values)
alpha_hat = nb_full.params["alpha"]
var_full = mu_full + alpha_hat * mu_full ** 2
analysis_df["pearson_resid"] = (y.values - mu_full) / np.sqrt(var_full)

constituencies = gpd.read_file(SHAPEFILE_PATH)
resid_map_df = constituencies[["WKR_NR", "geometry"]].rename(columns={"WKR_NR": "constituency_code"}).merge(
    analysis_df[["constituency_code", "pearson_resid"]], on="constituency_code", how="left"
).dropna(subset=["pearson_resid"]).reset_index(drop=True)

w = Queen.from_dataframe(resid_map_df, use_index=False)
w.transform = "r"
moran_resid = Moran(resid_map_df["pearson_resid"].values, w, permutations=999)

print(f"Moran's I on NB2 Pearson residuals = {moran_resid.I:.4f}")
print(f"p-value (999 permutations) = {moran_resid.p_sim:.4f}")
print(f"z-score = {moran_resid.z_sim:.2f}")
print(f"\nFor comparison, Moran's I on the raw racialised_rate (\u00a72.6) = 0.28, p = 0.001")

In [ ]:
# Task 1 settled on keeping pool_count as the primary specification (§3.4b), so the
# check above (on nb_full's residuals) remains the primary result. For completeness,
# we also check whether the density-only (log) variant's residuals show materially
# different spatial clustering -- i.e. whether the choice of density covariate matters
# for the unexplained spatial structure, not just for in-sample/CV fit.
mu_density = np.exp(X_v_density.values @ nb_v_density.params.drop("alpha").values + offset.values)
alpha_density = nb_v_density.params["alpha"]
var_density = mu_density + alpha_density * mu_density ** 2
resid_density = (y.values - mu_density) / np.sqrt(var_density)

resid_map_density = constituencies[["WKR_NR", "geometry"]].rename(columns={"WKR_NR": "constituency_code"}).merge(
    analysis_df[["constituency_code"]].assign(pearson_resid=resid_density), on="constituency_code", how="left"
).dropna(subset=["pearson_resid"]).reset_index(drop=True)

moran_density = Moran(resid_map_density["pearson_resid"].values, w, permutations=999)
print(f"Moran's I, pool_count-based model residuals (primary, above)  = {moran_resid.I:.4f} "
      f"(p={moran_resid.p_sim:.3f})")
print(f"Moran's I, density-only (log) model residuals                 = {moran_density.I:.4f} "
      f"(p={moran_density.p_sim:.3f})")

**Takeaway (density-covariate comparison).** The density-only variant's residual Moran's I (~0.11) is essentially the same as the pool_count-based model's (0.118) -- the two differ by less than either model's own permutation noise would allow us to distinguish. Swapping which density covariate goes into the model does not meaningfully change the amount of residual spatial clustering left over; the unexplained spatial structure is a property of the outcome and the predictor set as a whole (most likely something like a state-level effect or genuine spatial autocorrelation in unmeasured factors), not an artifact of choosing `pool_count` over `population_density_per_km2`. This reinforces §3.6's original conclusion that a spatial-lag/error term or state random effect is the right next step, independent of which density measure the model eventually settles on.

**Takeaway.** Moran's I on the model's Pearson residuals is 0.118 (p=0.004, z=3.50) — still significant, but less than half of the raw `racialised_rate`'s I=0.28. So this model explains part of the spatial clustering §2.6 found (mainly through `foreign_population_pct`, which is itself spatially structured — the Ruhr and Berlin hotspots are both higher-foreign-population-share areas), but a meaningful amount of spatial structure remains unexplained. This is the clearest concrete item for the next iteration: a spatial-lag/spatial-error NB2, or a state-level random effect (cheaper to implement), rather than adding more socioeconomic covariates to this specification.

## 3.7 Part 3 synthesis: what this preliminary model shows

**What this preliminary model supports:**
- A genuine, now triply-confirmed effect of `foreign_population_pct` on the racialised-review rate, net of venue density and five other predictors (IRR=1.034, p<0.0001 in the primary `pool_count`-based spec) -- the single most robust quantitative finding across all three notebooks. §3.4b confirms this holds up (IRR 1.026-1.034, p<0.05 in every case) whether the density covariate is `pool_count`, `population_density_per_km2` (log), or both together -- it is not an artifact of which density measure is chosen.
- `pool_count` remains the recommended density covariate, but only marginally. §3.4b tested `population_density_per_km2` as an alternative, motivated by `02_socioeconomic_political_relations.ipynb`'s finding that density is the more fundamental driver in raw correlation terms (density vs `racialised_rate` ρ=0.597 vs `pool_count`'s ρ=-0.556, with the two strongly collinear at ρ=-0.794). Once folded into this notebook's full multivariate NB2 alongside `foreign_population_pct` and the other five predictors, that raw-correlation advantage does not carry over: `pool_count` still edges out density-only and both-together on AIC (2200.6 vs 2202.8 vs 2201.6), BIC, and 5-fold CV MAE (0.610pp vs 0.613pp vs 0.615pp), and LR tests confirm adding the other density measure to either single-covariate model does not earn its extra parameter (p=0.32 and p=0.074 respectively). The margins are small, so this is a marginal, revisable call, not a decisive rejection of density.
- The 6-predictor set as a whole adds real explanatory power over `pool_count` alone (LR=77.2, p=1.3e-14) and improves out-of-sample prediction (5-fold CV MAE 0.61pp vs. 0.69pp) -- a modest but genuine predictive gain.
- Coefficients are stable under the `reliable_n` robustness check (§3.5) -- now correctly **247/299** constituencies (corrected from a stale 288/299 in this section, which predated `descriptive_analysis.ipynb`'s tightening of the reliability bar from `racialised_reviews>=5` to `>=10`). All 7 coefficients keep their sign on the stricter 247-constituency subset; the largest shifts (`unemployment_total` -0.039, `sgb_ii_recipients_pct` +0.026) are larger than the old check reported, consistent with a bigger, more different set of excluded constituencies (52 now vs. 11 before), but still well within the range expected from sampling noise given the coefficients' standard errors in §3.3.

**What this preliminary model does not (yet) support, and why:**
- Individual effects of `sgb_ii_recipients_pct` and `unemployment_total` -- both dip below significance once modelled jointly (VIF 26.8/25.8, §3.2), because they're too collinear with each other to separate. §2.4b's partial correlations should be read as evidence of a shared "deprivation" signal here, not as two independent effects. (Notebook 02 also flags that 20 constituencies -- Berlin 12, Hamburg 6, Bremen 2 -- share state-wide-constant values on several of the ingredient columns behind these two predictors and others; this is a data-granularity caveat rather than a bug, and it applies to this model too. It did not warrant re-running the full robustness suite excluding those 20 here, since notebook 02 found no sign flips and only modest (up to 0.09) shifts when doing so at the correlation level.)
- `turnout_pct`, `second_vote_spd_pct`, `second_vote_bsw_pct` -- all non-significant once placed alongside the stronger predictors; their §2.4b partial correlations were real but too small to survive joint estimation at n=299.
- A model that fully explains the spatial clustering §2.6 found: residual Moran's I=0.12 (p=0.004) is roughly half the raw rate's I=0.28, meaning genuine spatial structure remains after controlling for these six predictors and venue density. §3.6's extended check confirms this is not sensitive to the `pool_count`-vs-density choice either (density-only variant's residual I~0.11 is statistically indistinguishable from the primary model's).
- A model using all 18 §2.4 predictors together, tried and explicitly rejected in §3.2 for this pass (VIF up to 79.7, MLE fails to converge) -- a full accounting of every predictor isn't viable at n=299 without either dimensionality reduction (e.g. a deprivation composite index) or a different estimation approach (e.g. regularisation).
- A joint `pool_count` + `population_density_per_km2` specification (§3.4b): technically estimable (VIF 7.24, under the conventional danger threshold) but not supported by the LR tests, so not adopted. If `population_density_per_km2` is used in a future iteration, note that **the log-transform is a permanent requirement, not just a diagnostic check for this round** -- the raw scale is skewed enough (skew~3.3 vs. `pool_count`'s ~0.9) that the untransformed NB2 fails to converge outright.

**Concrete next-iteration items, in priority order:** (1) a spatial-lag/error NB2 or state-level random effect to address the residual clustering in §3.6, which §3.4b confirms is present regardless of the density covariate used; (2) resolve the `sgb_ii_recipients_pct`/`unemployment_total` collinearity, either by combining them into one deprivation index or picking one on theoretical grounds; (3) revisit whether `population_density_per_km2` should replace `pool_count` once a further iteration either resolves the density/pool_count near-tie more decisively or brings in additional predictors that might interact differently with a true density measure than with a venue count; (4) revisit once `theme`/`approach` IAA is finalised, since the outcome label itself (`racialised`) is what all of this is built on. This is a first, deliberately simple predictive pass, consistent with the plan agreed before starting -- the model, predictor set, and diagnostics above are all expected to be revised as that work continues.